In [ ]:
# Memasang pustaka untuk pengolahan data, model, evaluasi, dan visualisasi.
%pip install -q "transformers>=4.52,<5" "peft>=0.16,<0.20" "accelerate>=1,<2" "bitsandbytes>=0.46" "imbalanced-learn>=0.13" "scikit-learn>=1.4" pandas numpy matplotlib tqdm

In [ ]:
import gc  # Mengelola pembersihan memori Python.
import time  # Mengukur durasi pelatihan dan validasi setiap epoch.
import json  # Menyimpan dan membaca metadata dalam format JSON.
import random  # Mengendalikan pengacakan Python.
import shutil  # Menyalin dataset dan berkas checkpoint.
import hashlib  # Membuat identitas untuk konfigurasi eksperimen.
from pathlib import Path  # Menyusun lokasi file dan folder.

import numpy as np  # Mengolah data numerik.
import pandas as pd  # Mengolah dataset berbentuk tabel.
import torch  # Menjalankan pelatihan menggunakan GPU.
import matplotlib.pyplot as plt  # Membuat grafik untuk Bab IV.
from IPython.display import display  # Menampilkan tabel di Google Colab.
from google.colab import drive  # Menghubungkan Colab dengan Google Drive.
from sklearn.model_selection import train_test_split  # Mengambil sampel dengan stratifikasi kelas.
from sklearn.model_selection import GroupShuffleSplit  # Memisahkan kelompok profil tanpa tumpang tindih.
from sklearn.metrics import accuracy_score  # Menghitung akurasi klasifikasi.
from sklearn.metrics import f1_score  # Menghitung F1-score.
from sklearn.metrics import precision_recall_fscore_support  # Menghitung metrik khusus kelas positif.
from sklearn.metrics import classification_report  # Membuat laporan seluruh kelas.
from sklearn.metrics import confusion_matrix  # Menghitung matriks kebingungan.
from sklearn.metrics import ConfusionMatrixDisplay  # Menggambar matriks kebingungan.
from imblearn.under_sampling import RandomUnderSampler  # Menyeimbangkan data latih dengan RUS.

drive.mount('/content/drive')  # Meminta akses ke Google Drive.
ROOT = Path('/content/drive/MyDrive')  # Menunjuk lokasi utama My Drive.
FILE = ROOT / 'LLCP2024.XPT_'  # Menentukan nama dataset dengan garis bawah di akhir.
OUT = ROOT / 'Hasil_Tesis_TinyLlama_87500_Rerun_Terstruktur'  # Menentukan folder keluaran penelitian.
OUT.mkdir(parents=True, exist_ok=True)  # Membuat folder keluaran apabila belum ada.

SEED = 42  # Menetapkan angka acak agar eksperimen dapat diulang.
JUMLAH_KANDIDAT = 120  # Memeriksa beberapa pembagian profil agar distribusi kelas lebih serupa.
N_SAMPEL = 87500  # Menentukan jumlah responden yang digunakan dalam eksperimen.
EPOCHS = 5  # Menentukan jumlah maksimum putaran pelatihan.
BATCH = 8  # Menentukan jumlah sampel per batch pelatihan.
BATCH_EVAL = 24  # Menentukan jumlah sampel per batch evaluasi.
MAX_TOKEN = 128  # Menentukan batas panjang token berdasarkan rancangan Bab III.
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Menentukan model dasar untuk kedua eksperimen.

random.seed(SEED)  # Menetapkan seed untuk fungsi acak Python.
np.random.seed(SEED)  # Menetapkan seed untuk NumPy.
torch.manual_seed(SEED)  # Menetapkan seed untuk PyTorch.

assert FILE.is_file(), f'File dataset tidak ditemukan: {FILE}'  # Memastikan file XPT tersedia di My Drive.
assert torch.cuda.is_available(), 'Aktifkan GPU melalui Runtime > Change runtime type.'  # Memastikan GPU sudah aktif.

print('GPU:', torch.cuda.get_device_name(0))  # Menampilkan nama GPU yang digunakan.
print('Dataset:', FILE)  # Menampilkan lokasi dataset sumber.
print('Folder hasil:', OUT)  # Menampilkan lokasi penyimpanan hasil.

In [ ]:
from importlib.metadata import version  # Membaca versi pustaka yang benar-benar terpasang.
from importlib.metadata import PackageNotFoundError  # Menangani pustaka yang tidak terdaftar.
import platform  # Membaca versi Python yang digunakan.
import sys  # Membaca informasi runtime Python.

NAMA_PUSTAKA = [  # Menentukan semua pustaka yang penting bagi replikasi eksperimen.
    'torch',  # Membaca versi pustaka pelatihan model.
    'pandas',  # Membaca versi pustaka manipulasi tabel.
    'numpy',  # Membaca versi pustaka komputasi numerik.
    'transformers',  # Membaca versi pustaka model TinyLlama.
    'peft',  # Membaca versi pustaka adapter QLoRA.
    'bitsandbytes',  # Membaca versi pustaka kuantisasi model.
    'accelerate',  # Membaca versi pustaka dukungan komputasi.
    'scikit-learn',  # Membaca versi pustaka pembagian dan evaluasi data.
    'imbalanced-learn',  # Membaca versi pustaka RUS.
    'matplotlib',  # Membaca versi pustaka visualisasi.
    'tqdm',  # Membaca versi pustaka penampil progres.
]  # Mengakhiri daftar pustaka yang diperiksa.

catatan_versi = []  # Menyediakan tempat untuk nama dan versi pustaka.
for nama in NAMA_PUSTAKA:  # Memeriksa satu per satu pustaka dalam lingkungan saat ini.
    try:  # Mencoba mendapatkan versi pustaka yang terpasang.
        versi = version(nama)  # Mengambil nomor versi melalui metadata paket.
    except PackageNotFoundError:  # Menangani pustaka yang tidak ditemukan.
        versi = 'Tidak terpasang'  # Menandai pustaka yang tidak tersedia.
    catatan_versi.append({'Pustaka': nama, 'Versi': versi})  # Mencatat informasi versi.

versi_pustaka = pd.DataFrame(catatan_versi)  # Menyusun hasil pemeriksaan menjadi tabel.
versi_pustaka.to_csv(OUT / 'versi_pustaka_aktual.csv', index=False)  # Menyimpan versi untuk Tabel 3.1 tesis.
display(versi_pustaka)  # Menampilkan versi pustaka agar dapat diperiksa.

lingkungan = {  # Menyusun identitas komputer dan perangkat lunak eksperimen.
    'python': platform.python_version(),  # Mencatat versi Python yang sedang berjalan.
    'pytorch': torch.__version__,  # Mencatat versi PyTorch.
    'cuda_pytorch': torch.version.cuda,  # Mencatat versi CUDA yang terkait dengan PyTorch.
    'gpu': torch.cuda.get_device_name(0),  # Mencatat jenis GPU yang digunakan.
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2),  # Mencatat kapasitas memori GPU.
    'versi_pustaka': dict(zip(versi_pustaka['Pustaka'], versi_pustaka['Versi'])),  # Menggabungkan semua versi paket.
}  # Mengakhiri metadata lingkungan komputasi.

(OUT / 'lingkungan_komputasi.json').write_text(json.dumps(lingkungan, indent=2))  # Menyimpan lingkungan dalam JSON.
print('Python:', lingkungan['python'])  # Menampilkan versi Python yang dijalankan.
print('GPU:', lingkungan['gpu'])  # Menampilkan nama GPU yang digunakan.
print('VRAM (GB):', lingkungan['vram_gb'])  # Menampilkan kapasitas GPU.

In [ ]:
from matplotlib.ticker import FuncFormatter  # Mengatur tampilan bilangan pada sumbu grafik.

WARNA_GELAP = '#123B66'  # Menentukan biru tua untuk kelompok utama.
WARNA_SEDANG = '#2874B2'  # Menentukan biru sedang untuk kelompok pembanding.
WARNA_MUDA = '#79ADD1'  # Menentukan biru muda untuk kelompok tambahan.
WARNA_PUCAT = '#C4DDED'  # Menentukan biru pucat untuk bagian pelengkap.
PALET = [WARNA_GELAP, WARNA_SEDANG, WARNA_MUDA, WARNA_PUCAT]  # Menyatukan palet penelitian.
NAMA_KELAS = ['No CVD History', 'CVD History']  # Menetapkan nama kelas asli penelitian.
UKURAN_GAMBAR = (8, 5)  # Menyamakan ukuran semua grafik menjadi 8 × 5 inci.
DPI = 200  # Menyamakan resolusi gambar menjadi 1600 × 1000 piksel.
FOLDER_GAMBAR = OUT / 'Gambar_Bab_IV'  # Menentukan folder khusus seluruh gambar untuk Bab IV.
FOLDER_GAMBAR.mkdir(parents=True, exist_ok=True)  # Membuat folder gambar bila belum tersedia.

plt.rcParams.update({  # Menerapkan pengaturan tipografi dan sumbu pada semua grafik.
    'font.family': 'DejaVu Sans',  # Memilih jenis huruf yang tersedia di Colab.
    'font.size': 10,  # Menentukan ukuran teks dasar.
    'axes.labelsize': 11,  # Menyamakan ukuran label sumbu.
    'xtick.labelsize': 9,  # Menentukan ukuran angka atau kategori pada sumbu horizontal.
    'ytick.labelsize': 9,  # Menentukan ukuran angka atau kategori pada sumbu vertikal.
    'legend.fontsize': 9,  # Menyamakan ukuran legenda seluruh gambar.
    'axes.edgecolor': '#B5C7D9',  # Menggunakan garis sumbu yang lembut.
    'axes.spines.top': False,  # Menghilangkan garis bingkai bagian atas.
    'axes.spines.right': False,  # Menghilangkan garis bingkai bagian kanan.
    'figure.facecolor': 'white',  # Menetapkan latar kanvas putih.
    'axes.facecolor': 'white',  # Menetapkan latar grafik putih.
    'savefig.facecolor': 'white',  # Memastikan latar gambar tersimpan tetap putih.
})  # Menutup pengaturan gaya Matplotlib.

def kanvas(kiri=0.16, bawah=0.18, kanan=0.96, atas=0.94):  # Membuat kanvas grafik yang seragam.
    fig, ax = plt.subplots(figsize=UKURAN_GAMBAR)  # Menyiapkan gambar 8 × 5 inci.
    fig.subplots_adjust(left=kiri, bottom=bawah, right=kanan, top=atas)  # Mengatur margin secara konsisten.
    ax.set_axisbelow(True)  # Menempatkan kisi di belakang nilai grafik.
    ax.grid(axis='y', color='#E4EDF5', linewidth=0.7)  # Menggunakan garis bantu yang halus.
    return fig, ax  # Mengembalikan kanvas untuk pembuatan grafik.

def simpan_gambar(fig, nama):  # Menyimpan setiap gambar dengan ukuran dan nama tetap.
    fig.savefig(FOLDER_GAMBAR / f'{nama}.png', dpi=DPI)  # Menyimpan PNG 1600 × 1000 piksel.
    fig.savefig(FOLDER_GAMBAR / f'{nama}.svg')  # Menyimpan SVG agar tajam saat ditempel di Word.
    plt.show()  # Menampilkan gambar setelah tahapan terkait selesai.
    plt.close(fig)  # Menutup gambar untuk menghemat memori.

def format_ribuan(nilai, posisi):  # Menyesuaikan format jumlah menurut penulisan Indonesia.
    return f'{nilai:,.0f}'.replace(',', '.')  # Menggunakan titik sebagai pemisah ribuan.

In [ ]:
FITUR = [  # Menentukan sebelas variabel prediktor BRFSS 2024.
    '_AGEG5YR',  # Menggunakan kategori kelompok usia.
    '_SEX',  # Menggunakan jenis kelamin.
    '_BMI5CAT',  # Menggunakan kategori indeks massa tubuh.
    'GENHLTH',  # Menggunakan penilaian kesehatan umum.
    'DIABETE4',  # Menggunakan status riwayat diabetes.
    'CHCKDNY2',  # Menggunakan riwayat penyakit ginjal.
    'CVDSTRK3',  # Menggunakan riwayat stroke.
    'CHCCOPD3',  # Menggunakan riwayat COPD.
    '_RFSMOK3',  # Menggunakan indikator status merokok.
    '_TOTINDA',  # Menggunakan indikator aktivitas fisik.
    '_RFBING6',  # Menggunakan indikator binge drinking.
]  # Mengakhiri daftar fitur terpilih.

KOLOM = FITUR + ['_MICHD']  # Menambahkan variabel target hanya untuk pembentukan label.
LOKAL = Path('/content/LLCP2024.XPT_')  # Menentukan salinan dataset di penyimpanan sementara Colab.

if not LOKAL.exists():  # Memeriksa apakah salinan lokal sudah tersedia.
    shutil.copy2(FILE, LOKAL)  # Menyalin XPT dari Drive agar pembacaan lebih cepat.

potongan = []  # Menyediakan penampung bagian dataset terpilih.
jumlah_awal = 0  # Menghitung seluruh rekaman dari file sumber.
jumlah_kolom = None  # Menyimpan jumlah variabel dalam dataset sumber.

pembaca = pd.read_sas(  # Membuka dataset SAS dalam beberapa potongan.
    LOKAL,  # Mengambil sumber dari salinan lokal.
    format='xport',  # Menetapkan format file SAS XPORT.
    chunksize=20000,  # Membatasi jumlah baris yang dibaca dalam satu potongan.
)  # Mengakhiri pengaturan pembacaan XPT.

for bagian in pembaca:  # Membaca potongan dataset satu demi satu.
    if jumlah_kolom is None:  # Memeriksa apakah jumlah kolom sumber belum tercatat.
        jumlah_kolom = len(bagian.columns)  # Menghitung variabel dalam dataset asli.
    jumlah_awal += len(bagian)  # Menambahkan jumlah responden pada potongan sekarang.
    potongan.append(bagian[KOLOM].copy())  # Mengambil hanya sebelas fitur dan satu target.

data = pd.concat(potongan, ignore_index=True)  # Menggabungkan potongan menjadi dataset kerja.
del potongan  # Menghapus penampung bagian dataset yang tidak lagi diperlukan.
gc.collect()  # Membebaskan memori dari objek sementara.

print('Jumlah data awal:', jumlah_awal)  # Menampilkan jumlah seluruh responden.
print('Jumlah variabel awal:', jumlah_kolom)  # Menampilkan jumlah variabel sumber.
print('Ukuran data terpilih:', data.shape)  # Menampilkan baris dan kolom setelah seleksi.

In [ ]:
VALID = {  # Mendefinisikan kode kategori valid berdasarkan codebook BRFSS 2024.
    '_AGEG5YR': range(1, 14),  # Mempertahankan kelompok usia 1 sampai 13.
    '_SEX': [1, 2],  # Mempertahankan dua kode jenis kelamin.
    '_BMI5CAT': [1, 2, 3, 4],  # Mempertahankan empat kategori BMI.
    'GENHLTH': [1, 2, 3, 4, 5],  # Mempertahankan lima tingkat kesehatan umum.
    'DIABETE4': [1, 2, 3, 4],  # Mempertahankan empat status diabetes yang valid.
    'CHCKDNY2': [1, 2],  # Mempertahankan jawaban ya atau tidak untuk penyakit ginjal.
    'CVDSTRK3': [1, 2],  # Mempertahankan jawaban ya atau tidak untuk stroke.
    'CHCCOPD3': [1, 2],  # Mempertahankan jawaban ya atau tidak untuk COPD.
    '_RFSMOK3': [1, 2],  # Mempertahankan dua kategori indikator merokok.
    '_TOTINDA': [1, 2],  # Mempertahankan dua kategori aktivitas fisik.
    '_RFBING6': [1, 2],  # Mempertahankan dua kategori binge drinking.
    '_MICHD': [1, 2],  # Mempertahankan status riwayat CHD/MI yang valid.
}  # Mengakhiri definisi kategori valid.

for kolom in KOLOM:  # Memeriksa semua fitur dan target yang dipilih.
    nilai_valid = VALID[kolom]  # Mengambil kode yang diizinkan untuk satu variabel.
    data[kolom] = data[kolom].where(data[kolom].isin(nilai_valid))  # Mengganti nilai tidak valid dengan NaN.

invalid = data[KOLOM].isna().sum()  # Menghitung jumlah nilai kosong atau tidak valid setiap kolom.
invalid = invalid.sort_values(ascending=False)  # Mengurutkan variabel berdasarkan jumlah nilai tidak valid.
invalid.to_csv(OUT / 'nilai_tidak_valid_per_variabel.csv', header=['jumlah'])  # Menyimpan audit kode tidak valid.

bersih = data.dropna(subset=KOLOM).copy()  # Menghapus responden yang tidak valid pada fitur atau target.
bersih[KOLOM] = bersih[KOLOM].astype('int16')  # Mengubah kategori menjadi bilangan bulat hemat memori.
bersih['label'] = (bersih['_MICHD'] == 1).astype('int8')  # Memberi label 1 untuk riwayat CHD/MI dan 0 untuk tidak.

jumlah_dihapus = len(data) - len(bersih)  # Menghitung jumlah responden yang keluar saat preprocessing.
print('Jumlah data bersih:', len(bersih))  # Menampilkan jumlah responden yang valid.
print('Jumlah data dikeluarkan:', jumlah_dihapus)  # Menampilkan jumlah responden yang dikeluarkan.
print(bersih['label'].value_counts().sort_index())  # Menampilkan distribusi kelas pada data bersih.

In [ ]:
awal_label = data['_MICHD'].map({1: 1, 2: 0})  # Mengambil label target valid sebelum pembersihan fitur.
hitung_awal = awal_label.value_counts().reindex([0, 1], fill_value=0)  # Menghitung jumlah tiap kelas sebelum pembersihan.
hitung_bersih = bersih['label'].value_counts().reindex([0, 1], fill_value=0)  # Menghitung jumlah tiap kelas setelah pembersihan.
tabel_preprocessing = pd.DataFrame({'Sebelum pembersihan': hitung_awal, 'Setelah pembersihan': hitung_bersih})  # Membandingkan jumlah kelas.
tabel_preprocessing.index = NAMA_KELAS  # Mengganti kode label dengan nama kelas.
tabel_preprocessing.to_csv(OUT / 'kelas_sebelum_sesudah_preprocessing.csv')  # Menyimpan data grafik dalam CSV.

fig, ax = kanvas(bawah=0.20)  # Membuat kanvas untuk distribusi kelas.
posisi = np.arange(2)  # Menentukan posisi dua kategori pada sumbu horizontal.
lebar = 0.34  # Menentukan lebar masing-masing kelompok batang.
ax.bar(posisi - lebar / 2, tabel_preprocessing.iloc[:, 0], width=lebar, color=WARNA_GELAP, label='Sebelum pembersihan')  # Menampilkan data awal.
ax.bar(posisi + lebar / 2, tabel_preprocessing.iloc[:, 1], width=lebar, color=WARNA_MUDA, label='Setelah pembersihan')  # Menampilkan data bersih.
ax.set_xticks(posisi, NAMA_KELAS)  # Menuliskan kelas pada sumbu horizontal.
ax.set_ylabel('Jumlah responden')  # Menjelaskan jumlah pada sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Menampilkan titik ribuan pada sumbu.
ax.legend(frameon=False)  # Menampilkan legenda tanpa kotak tambahan.
simpan_gambar(fig, '01_distribusi_kelas_preprocessing')  # Menampilkan dan menyimpan hasil preprocessing.

In [ ]:
awal_label = data['_MICHD'].map({1: 1, 2: 0})  # Mengambil label target valid sebelum pembersihan fitur.
hitung_awal = awal_label.value_counts().reindex([0, 1], fill_value=0)  # Menghitung jumlah tiap kelas sebelum pembersihan.
hitung_bersih = bersih['label'].value_counts().reindex([0, 1], fill_value=0)  # Menghitung jumlah tiap kelas setelah pembersihan.
tabel_preprocessing = pd.DataFrame({'Sebelum pembersihan': hitung_awal, 'Setelah pembersihan': hitung_bersih})  # Membandingkan jumlah kelas.
tabel_preprocessing.index = NAMA_KELAS  # Mengganti kode label dengan nama kelas.
tabel_preprocessing.to_csv(OUT / 'kelas_sebelum_sesudah_preprocessing.csv')  # Menyimpan data grafik dalam CSV.

fig, ax = kanvas(bawah=0.20)  # Membuat kanvas untuk distribusi kelas.
posisi = np.arange(2)  # Menentukan posisi dua kategori pada sumbu horizontal.
lebar = 0.34  # Menentukan lebar masing-masing kelompok batang.

# Menggambar data awal dan bersih, lalu simpan objek batangnya
bars_sebelum = ax.bar(posisi - lebar / 2, tabel_preprocessing.iloc[:, 0], width=lebar, color=WARNA_GELAP, label='Sebelum pembersihan')
bars_sesudah = ax.bar(posisi + lebar / 2, tabel_preprocessing.iloc[:, 1], width=lebar, color=WARNA_MUDA, label='Setelah pembersihan')

# Membuat format teks dengan titik ribuan secara manual
labels_sebelum = [f'{int(val):,}'.replace(',', '.') for val in tabel_preprocessing.iloc[:, 0]]
labels_sesudah = [f'{int(val):,}'.replace(',', '.') for val in tabel_preprocessing.iloc[:, 1]]

# Menambahkan label nilai di atas masing-masing batang
ax.bar_label(bars_sebelum, labels=labels_sebelum, padding=3, fontsize=8, rotation=0)
ax.bar_label(bars_sesudah, labels=labels_sesudah, padding=3, fontsize=8, rotation=0)

ax.set_xticks(posisi, NAMA_KELAS)  # Menuliskan kelas pada sumbu horizontal.
ax.set_ylabel('Jumlah responden')  # Menjelaskan jumlah pada sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Menampilkan titik ribuan pada sumbu.

# Menyesuaikan batas sumbu Y secara otomatis agar ada ruang untuk teks di atas batang tertinggi
max_prep = tabel_preprocessing.max().max()
ax.set_ylim(0, max_prep * 1.20)

ax.legend(frameon=False)  # Menampilkan legenda tanpa kotak tambahan.
simpan_gambar(fig, '01_distribusi_kelas_preprocessing')  # Menampilkan dan menyimpan hasil preprocessing.

In [ ]:
fig, ax = kanvas(kiri=0.23, bawah=0.15)  # Membuat bidang grafik nilai tidak valid.
invalid_urut = invalid.sort_values(ascending=True)  # Mengurutkan jumlah nilai tidak valid dari kecil ke besar.
ax.barh(invalid_urut.index, invalid_urut.values, color=WARNA_SEDANG)  # Menampilkan jumlah nilai tidak valid per variabel.
ax.set_xlabel('Jumlah nilai tidak valid')  # Menuliskan arti sumbu horizontal.
ax.grid(axis='x', color='#E4EDF5', linewidth=0.7)  # Memindahkan kisi ke sumbu jumlah.
ax.grid(axis='y', visible=False)  # Menghapus kisi yang mengganggu nama variabel.
simpan_gambar(fig, '02_nilai_tidak_valid_per_variabel')  # Menampilkan dan menyimpan audit nilai tidak valid.


In [ ]:
fig, ax = kanvas(kiri=0.23, bawah=0.15)  # Membuat bidang grafik nilai tidak valid.
invalid_urut = invalid.sort_values(ascending=True)  # Mengurutkan jumlah nilai tidak valid dari kecil ke besar.

# Menampilkan jumlah nilai tidak valid per variabel secara horizontal dan simpan objek batangnya
bars = ax.barh(invalid_urut.index, invalid_urut.values, color=WARNA_SEDANG)

# Membuat format teks dengan titik ribuan secara manual berdasarkan nilai terurut
labels_invalid = [f'{int(val):,}'.replace(',', '.') for val in invalid_urut.values]

# Menambahkan label nilai di ujung kanan masing-masing batang horizontal
ax.bar_label(bars, labels=labels_invalid, padding=5, fontsize=9, fontweight='bold')

ax.set_xlabel('Jumlah nilai tidak valid')  # Menuliskan arti sumbu horizontal.
ax.grid(axis='x', color='#E4EDF5', linewidth=0.7)  # Memindahkan kisi ke sumbu jumlah.
ax.grid(axis='y', visible=False)  # Menghapus kisi yang mengganggu nama variabel.

# Menyesuaikan batas sumbu X secara otomatis agar ada ruang untuk teks di sebelah kanan batang terpanjang
max_invalid = invalid_urut.max()
ax.set_xlim(0, max_invalid * 1.25)

simpan_gambar(fig, '02_nilai_tidak_valid_per_variabel')  # Menampilkan dan menyimpan audit nilai tidak valid.

In [ ]:
profil_berulang_penuh = bersih.duplicated(subset=FITUR).sum()  # Menghitung baris dengan profil sebelas fitur berulang.
profil_label_berulang_penuh = bersih.duplicated(subset=FITUR + ['label']).sum()  # Menghitung pengulangan profil sekaligus label.
assert len(bersih) >= N_SAMPEL, 'Jumlah data valid kurang dari jumlah sampel.'  # Memastikan sampel berukuran 87.500 dapat diambil.

sampel, sisa_tidak_terpilih = train_test_split(  # Mengambil sampel dengan rasio kelas yang tetap mendekati data bersih.
    bersih,  # Menggunakan data yang telah lolos preprocessing.
    train_size=N_SAMPEL,  # Memilih tepat 87.500 responden.
    stratify=bersih['label'],  # Menjaga proporsi kelas saat pengambilan sampel.
    random_state=SEED,  # Menetapkan pengambilan sampel yang dapat direproduksi.
)  # Mengakhiri pengambilan sampel terstratifikasi.

del sisa_tidak_terpilih  # Menghapus salinan responden yang tidak digunakan.
sampel = sampel.reset_index(drop=True).copy()  # Menetapkan indeks berurutan pada sampel.
sampel['id_profil'] = sampel.groupby(FITUR, sort=False).ngroup()  # Memberi ID yang sama untuk kombinasi fitur identik.

jumlah_profil = sampel['id_profil'].nunique()  # Menghitung jumlah kombinasi fitur yang berbeda.
ulang_fitur = sampel.duplicated(subset=FITUR).sum()  # Menghitung baris dengan profil yang berulang dalam sampel.
ulang_fitur_label = sampel.duplicated(subset=FITUR + ['label']).sum()  # Menghitung pengulangan profil bersama label.
konflik_label = sampel.groupby('id_profil')['label'].nunique().gt(1).sum()  # Menghitung profil sama yang memiliki label berbeda.

audit_nama = [  # Menentukan nama indikator audit yang akan dilaporkan.
    'Data bersih',  # Menunjukkan jumlah responden setelah preprocessing.
    'Profil berulang pada data bersih',  # Menunjukkan pengulangan kombinasi fitur di data penuh.
    'Fitur dan label berulang pada data bersih',  # Menunjukkan pengulangan kombinasi fitur dan target.
    'Sampel eksperimen',  # Menunjukkan jumlah responden yang digunakan pada eksperimen.
    'Profil unik di sampel',  # Menunjukkan variasi kombinasi fitur dalam sampel.
    'Profil berulang di sampel',  # Menunjukkan jumlah kemunculan profil berulang dalam sampel.
    'Fitur dan label berulang di sampel',  # Menunjukkan pengulangan fitur dan label dalam sampel.
    'Profil sampel dengan dua label',  # Menunjukkan kombinasi fitur dengan target yang bertentangan.
]  # Mengakhiri nama indikator audit.

audit_nilai = [  # Menyusun nilai indikator audit sesuai urutan namanya.
    len(bersih),  # Mengambil jumlah responden valid.
    int(profil_berulang_penuh),  # Mengambil jumlah pengulangan profil dari data bersih.
    int(profil_label_berulang_penuh),  # Mengambil jumlah pengulangan profil dan label dari data bersih.
    len(sampel),  # Mengambil jumlah sampel terpilih.
    int(jumlah_profil),  # Mengambil jumlah profil unik.
    int(ulang_fitur),  # Mengambil jumlah pengulangan profil dalam sampel.
    int(ulang_fitur_label),  # Mengambil jumlah pengulangan fitur dan label pada sampel.
    int(konflik_label),  # Mengambil jumlah profil dengan dua kelas berbeda.
]  # Mengakhiri nilai indikator audit.

audit = pd.DataFrame({'Indikator': audit_nama, 'Jumlah': audit_nilai})  # Membentuk tabel audit untuk Bab IV.
audit.to_csv(OUT / 'audit_profil_11_fitur.csv', index=False)  # Menyimpan hasil audit profil berulang.
sampel.to_csv(OUT / 'dataset_sampel_87500.csv', index=False)  # Menyimpan sampel yang benar-benar digunakan.
display(audit)  # Menampilkan tabel hasil audit.

distribusi_sampel = pd.DataFrame({  # Membandingkan jumlah kelas sebelum dan setelah sampling.
    'Data bersih': bersih['label'].value_counts(),  # Menghitung kelas pada seluruh data bersih.
    'Sampel 87500': sampel['label'].value_counts(),  # Menghitung kelas pada sampel eksperimen.
})  # Mengakhiri pembentukan tabel distribusi.

distribusi_sampel = distribusi_sampel.reindex([0, 1]).fillna(0).astype(int)  # Menetapkan urutan kelas 0 dan 1.
distribusi_sampel.index = ['No CVD History', 'CVD History']  # Menampilkan label kelas dalam tabel.
distribusi_sampel.loc['Total'] = distribusi_sampel.sum()  # Menambahkan jumlah responden untuk setiap dataset.
distribusi_sampel.to_csv(OUT / 'distribusi_kelas_sampling_87500.csv')  # Menyimpan distribusi kelas.
display(distribusi_sampel)  # Menampilkan distribusi kelas sebelum dan setelah sampling.


In [ ]:
tabel_proporsi = distribusi_sampel.drop(index='Total').copy()  # Memilih dua baris kelas tanpa total.
persen_proporsi = tabel_proporsi.div(distribusi_sampel.loc['Total'], axis=1) * 100  # Menghitung persentase setiap kelas.
persen_proporsi.to_csv(OUT / 'persentase_kelas_sampling.csv')  # Menyimpan angka persentase pada CSV.

fig, ax = kanvas(bawah=0.18)  # Membuat grafik proporsi kelas sebelum dan sesudah sampling.
posisi = np.arange(len(persen_proporsi.columns))  # Menentukan posisi dua kelompok dataset.
lebar = 0.34  # Menetapkan lebar dua jenis kelas.
ax.bar(posisi - lebar / 2, persen_proporsi.loc['No CVD History'].to_numpy(), width=lebar, color=WARNA_GELAP, label='No CVD History')  # Menggambar kelas mayoritas.
ax.bar(posisi + lebar / 2, persen_proporsi.loc['CVD History'].to_numpy(), width=lebar, color=WARNA_MUDA, label='CVD History')  # Menggambar kelas minoritas.
ax.set_xticks(posisi, ['Data bersih', 'Sampel 87.500'])  # Menuliskan kelompok pembanding.
ax.set_ylabel('Proporsi responden (%)')  # Menetapkan satuan persentase.
ax.set_ylim(0, 105)  # Menetapkan rentang persentase yang sama untuk kedua kelas.
ax.legend(frameon=False)  # Menampilkan legenda tanpa garis tepi.
simpan_gambar(fig, '03_proporsi_kelas_sampling')  # Menyimpan grafik sampling.


In [ ]:
tabel_proporsi = distribusi_sampel.drop(index='Total').copy()  # Memilih dua baris kelas tanpa total.
persen_proporsi = tabel_proporsi.div(distribusi_sampel.loc['Total'], axis=1) * 100  # Menghitung persentase setiap kelas.
persen_proporsi.to_csv(OUT / 'persentase_kelas_sampling.csv')  # Menyimpan angka persentase pada CSV.

fig, ax = kanvas(bawah=0.18)  # Membuat grafik proporsi kelas sebelum dan sesudah sampling.
posisi = np.arange(len(persen_proporsi.columns))  # Menentukan posisi dua kelompok dataset.
lebar = 0.34  # Menetapkan lebar dua jenis kelas.

# Menggambar kelas mayoritas dan minoritas, lalu simpan objek batangnya
bars_no_cvd = ax.bar(posisi - lebar / 2, persen_proporsi.loc['No CVD History'].to_numpy(), width=lebar, color=WARNA_GELAP, label='No CVD History')
bars_cvd = ax.bar(posisi + lebar / 2, persen_proporsi.loc['CVD History'].to_numpy(), width=lebar, color=WARNA_MUDA, label='CVD History')

# Menambahkan label nilai persentase di atas masing-masing batang (format 1 angka di belakang koma)
ax.bar_label(bars_no_cvd, fmt='%.1f%%', padding=3, fontsize=8, rotation=0)
ax.bar_label(bars_cvd, fmt='%.1f%%', padding=3, fontsize=8, rotation=0)

ax.set_xticks(posisi, ['Data bersih', 'Sampel 87.500'])  # Menuliskan kelompok pembanding.
ax.set_ylabel('Proporsi responden (%)')  # Menetapkan satuan persentase.
ax.set_ylim(0, 112)  # Menetapkan rentang persentase dengan ruang ekstra di atas agar label tidak terpotong.
ax.legend(frameon=False)  # Menampilkan legenda tanpa garis tepi.
simpan_gambar(fig, '03_proporsi_kelas_sampling')  # Menyimpan grafik sampling.

In [ ]:
profil_audit = audit.set_index('Indikator')['Jumlah']  # Mengambil angka audit profil dari tabel sebelumnya.
indikator = ['Sampel eksperimen', 'Profil unik di sampel', 'Profil berulang di sampel', 'Profil sampel dengan dua label']  # Memilih empat indikator profil.
jumlah = profil_audit.loc[indikator]  # Membaca nilai untuk keempat indikator audit.

fig, ax = kanvas(kiri=0.36, bawah=0.14)  # Memberikan ruang untuk nama indikator panjang.
ax.barh(indikator[::-1], jumlah.iloc[::-1], color=[WARNA_MUDA, WARNA_SEDANG, WARNA_GELAP, WARNA_PUCAT])  # Menampilkan angka audit profil.
ax.set_xlabel('Jumlah profil atau responden')  # Menjelaskan satuan pada sumbu horizontal.
ax.grid(axis='x', color='#E4EDF5', linewidth=0.7)  # Menggunakan kisi vertikal halus.
ax.grid(axis='y', visible=False)  # Menghindari garis pada nama kategori.
simpan_gambar(fig, '04_audit_profil')  # Menampilkan dan menyimpan grafik audit profil.


In [ ]:
profil_audit = audit.set_index('Indikator')['Jumlah']  # Mengambil angka audit profil dari tabel sebelumnya.
indikator = ['Sampel eksperimen', 'Profil unik di sampel', 'Profil berulang di sampel', 'Profil sampel dengan dua label']  # Memilih empat indikator profil.
jumlah = profil_audit.loc[indikator]  # Membaca nilai untuk keempat indikator audit.

fig, ax = kanvas(kiri=0.36, bawah=0.14)  # Memberikan ruang untuk nama indikator panjang.

# Menampilkan angka audit profil secara horizontal dan simpan objek batangnya
bars = ax.barh(indikator[::-1], jumlah.iloc[::-1], color=[WARNA_MUDA, WARNA_SEDANG, WARNA_GELAP, WARNA_PUCAT])

# Membuat format teks dengan titik ribuan secara manual (dibalik sesuai urutan sumbu y)
labels_audit = [f'{int(val):,}'.replace(',', '.') for val in jumlah.iloc[::-1]]

# Menambahkan label nilai di ujung kanan masing-masing batang horizontal (tanpa parameter va)
ax.bar_label(bars, labels=labels_audit, padding=5, fontsize=9, fontweight='bold')

ax.set_xlabel('Jumlah profil atau responden')  # Menjelaskan satuan pada sumbu horizontal.
ax.grid(axis='x', color='#E4EDF5', linewidth=0.7)  # Menggunakan kisi vertikal halus.
ax.grid(axis='y', visible=False)  # Menghindari garis pada nama kategori.

# Menyesuaikan batas sumbu X secara otomatis agar ada ruang untuk teks di sebelah kanan batang terpanjang
max_audit = jumlah.max()
ax.set_xlim(0, max_audit * 1.22)

simpan_gambar(fig, '04_audit_profil')  # Menampilkan dan menyimpan grafik audit profil.

In [ ]:
def pilih_pembagian_grup(df, proporsi_sisa, seed, prevalensi_acuan):  # Memilih pembagian yang seimbang tanpa mencampur profil.
    pembagi = GroupShuffleSplit(  # Membuat beberapa kandidat pembagian menurut kelompok.
        n_splits=JUMLAH_KANDIDAT,  # Menentukan banyak kandidat yang diperiksa.
        test_size=proporsi_sisa,  # Menetapkan proporsi kelompok sementara.
        random_state=seed,  # Menjaga hasil pemilihan tetap dapat diulang.
    )  # Menutup pengaturan pembagian profil.

    pilihan = None  # Menyimpan kandidat dengan skor distribusi terbaik.
    skor_terkecil = float('inf')  # Menetapkan skor awal yang lebih besar daripada kandidat mana pun.

    for indeks_utama, indeks_sisa in pembagi.split(df, groups=df['id_profil']):  # Menilai kandidat pembagian satu per satu.
        kelas_utama = df['label'].iloc[indeks_utama]  # Mengambil label pada bagian pertama.
        kelas_sisa = df['label'].iloc[indeks_sisa]  # Mengambil label pada bagian kedua.

        if kelas_utama.nunique() < 2:  # Menolak kandidat tanpa kedua kelas pada bagian pertama.
            continue  # Melanjutkan pemeriksaan kandidat lain.
        if kelas_sisa.nunique() < 2:  # Menolak kandidat tanpa kedua kelas pada bagian kedua.
            continue  # Melanjutkan pemeriksaan kandidat lain.

        rasio_sisa = len(indeks_sisa) / len(df)  # Mengukur proporsi responden, bukan jumlah kelompok.
        selisih_rasio = abs(rasio_sisa - proporsi_sisa)  # Mengukur selisih dari target ukuran.
        selisih_kelas = abs(kelas_sisa.mean() - prevalensi_acuan)  # Mengukur perbedaan proporsi kelas positif.
        selisih_kelas_utama = abs(kelas_utama.mean() - prevalensi_acuan)  # Memeriksa proporsi kelas pada bagian pertama.

        skor = 8 * selisih_rasio  # Memberi bobot tinggi pada kedekatan jumlah responden.
        skor += 10 * selisih_kelas  # Menjaga proporsi kelas positif pada bagian sisa.
        skor += 10 * selisih_kelas_utama  # Menjaga proporsi kelas positif pada bagian utama.

        if skor < skor_terkecil:  # Memilih kandidat jika kualitas pembagiannya lebih baik.
            skor_terkecil = skor  # Memperbarui skor pembagian terbaik.
            pilihan = (indeks_utama, indeks_sisa)  # Menyimpan posisi dua bagian terpilih.

    if pilihan is None:  # Memastikan tersedia kandidat pembagian yang memuat dua kelas.
        raise RuntimeError('Tidak ditemukan pembagian grup dengan dua kelas pada setiap subset.')  # Menghentikan pembagian yang tidak valid.

    return pilihan  # Mengembalikan indeks dua subset yang tidak berbagi profil.

prevalensi = sampel['label'].mean()  # Mencatat proporsi kelas CVD History dari sampel penelitian.

indeks_latih, indeks_sisa = pilih_pembagian_grup(  # Memilih pemisahan yang mendekati 70% dan 30% responden.
    df=sampel,  # Menggunakan sampel terstratifikasi berukuran 87.500.
    proporsi_sisa=0.30,  # Menetapkan target 30% responden untuk validasi dan pengujian.
    seed=SEED,  # Menggunakan seed utama eksperimen.
    prevalensi_acuan=prevalensi,  # Menargetkan proporsi positif yang serupa dengan sampel.
)  # Menutup pemilihan pembagian pertama.

latih = sampel.iloc[indeks_latih].copy()  # Menyiapkan kelompok pelatihan sebelum RUS.
sisa = sampel.iloc[indeks_sisa].copy()  # Menyiapkan calon data validasi dan pengujian.

indeks_val, indeks_uji = pilih_pembagian_grup(  # Memisahkan kelompok sisa menjadi validasi dan pengujian.
    df=sisa,  # Menggunakan bagian yang belum masuk data pelatihan.
    proporsi_sisa=0.50,  # Menargetkan pembagian dua bagian dengan jumlah hampir sama.
    seed=SEED + 1,  # Memakai seed yang tetap tetapi berbeda dari pembagian pertama.
    prevalensi_acuan=prevalensi,  # Menargetkan prevalensi kelas positif pada sampel utama.
)  # Menutup pemilihan pembagian kedua.

validasi = sisa.iloc[indeks_val].copy()  # Menentukan kelompok validasi.
uji = sisa.iloc[indeks_uji].copy()  # Menentukan kelompok pengujian akhir.

grup_latih = set(latih['id_profil'])  # Mengumpulkan identitas profil pada data latih.
grup_val = set(validasi['id_profil'])  # Mengumpulkan identitas profil pada data validasi.
grup_uji = set(uji['id_profil'])  # Mengumpulkan identitas profil pada data uji.

assert not (grup_latih & grup_val), 'Profil sama muncul pada latih dan validasi.'  # Memastikan profil tidak tersebar pada dua subset.
assert not (grup_latih & grup_uji), 'Profil sama muncul pada latih dan pengujian.'  # Memastikan profil latih tidak muncul pada data uji.
assert not (grup_val & grup_uji), 'Profil sama muncul pada validasi dan pengujian.'  # Memastikan profil validasi tidak muncul pada data uji.
assert len(latih) + len(validasi) + len(uji) == len(sampel), 'Jumlah sampel berubah setelah pembagian.'  # Memeriksa seluruh sampel tetap digunakan.
assert all(df['label'].nunique() == 2 for df in (latih, validasi, uji)), 'Ada subset yang hanya mempunyai satu kelas.'  # Memastikan semua subset mempunyai kedua label.

rus = RandomUnderSampler(  # Menyiapkan penyeimbangan khusus data pelatihan.
    sampling_strategy=1.0,  # Menyeimbangkan kedua kelas pada rasio 1:1.
    random_state=SEED,  # Mengendalikan pemilihan kelas mayoritas yang dikurangi.
)  # Menutup konfigurasi penyeimbangan RUS.

rus.fit_resample(  # Memilih indeks sampel latih yang dipertahankan melalui RUS.
    latih[FITUR],  # Menggunakan fitur dari data pelatihan saja.
    latih['label'],  # Menggunakan label dari data pelatihan saja.
)  # Menutup pelaksanaan RUS.

latih_rus = latih.iloc[rus.sample_indices_].copy()  # Membuat data pelatihan dengan dua kelas seimbang.

ukuran = pd.DataFrame(  # Menyusun tabel jumlah tiap kelas berdasarkan kelompok data.
    {  # Menyusun kelompok data dalam bentuk pasangan nama dan nilai.
        'Latih sebelum RUS': latih['label'].value_counts(),  # Menghitung kelas sebelum diseimbangkan.
        'Latih setelah RUS': latih_rus['label'].value_counts(),  # Menghitung kelas setelah diseimbangkan.
        'Validasi alami': validasi['label'].value_counts(),  # Menghitung kelas pada validasi alami.
        'Uji alami': uji['label'].value_counts(),  # Menghitung kelas pada pengujian alami.
    }
)  # Menutup penyusunan tabel distribusi.

ukuran = ukuran.T.reindex(columns=[0, 1], fill_value=0).astype(int)  # Menetapkan urutan label 0 dan 1.
ukuran.columns = ['No CVD History', 'CVD History']  # Memberi nama kelas yang jelas.
ukuran['Jumlah'] = ukuran.sum(axis=1)  # Menjumlahkan responden pada setiap kelompok.
ukuran['Persentase CVD (%)'] = (ukuran['CVD History'] / ukuran['Jumlah'] * 100).round(2)  # Mengukur prevalensi kelas positif tiap kelompok.
ukuran.to_csv(OUT / 'distribusi_split_group_aware_rus.csv')  # Menyimpan distribusi kelompok untuk Bab IV.
display(ukuran)  # Menampilkan jumlah data dan proporsi kedua kelas.

ringkasan_split = pd.DataFrame(  # Merangkum ukuran dan distribusi sebelum RUS.
    {  # Menyusun kelompok data dalam bentuk pasangan nama dan nilai.
        'Subset': [  # Menetapkan nama kelompok penelitian.
            'Latih',  # Nama bagian pelatihan.
            'Validasi',  # Nama bagian validasi.
            'Pengujian',  # Nama bagian pengujian.
        ],  # Menutup daftar nama bagian.
        'Jumlah': [  # Mencatat jumlah responden di tiap kelompok.
            len(latih),  # Jumlah responden latih sebelum RUS.
            len(validasi),  # Jumlah responden validasi.
            len(uji),  # Jumlah responden pengujian.
        ],  # Menutup daftar ukuran data.
        'Proporsi responden (%)': [  # Mengukur rasio jumlah responden.
            100 * len(latih) / len(sampel),  # Persentase responden latih.
            100 * len(validasi) / len(sampel),  # Persentase responden validasi.
            100 * len(uji) / len(sampel),  # Persentase responden pengujian.
        ],  # Menutup daftar rasio responden.
        'CVD History (%)': [  # Mengukur proporsi kelas CVD History.
            100 * latih['label'].mean(),  # Proporsi kelas positif pada data latih.
            100 * validasi['label'].mean(),  # Proporsi kelas positif pada data validasi.
            100 * uji['label'].mean(),  # Proporsi kelas positif pada data pengujian.
        ],  # Menutup daftar prevalensi.
    }
)  # Menutup tabel ringkasan pembagian.

ringkasan_split.to_csv(OUT / 'ringkasan_pembagian_aktual.csv', index=False)  # Menyimpan ukuran dan distribusi hasil split.
display(ringkasan_split.round(2))  # Menampilkan ringkasan yang dapat digunakan pada Bab IV.

kelompok_pemeriksaan = {  # Menetapkan data yang akan diperiksa distribusi kelasnya.
    'Latih': latih,  # Mengacu pada data pelatihan sebelum RUS.
    'Validasi': validasi,  # Mengacu pada data validasi tanpa RUS.
    'Pengujian': uji,  # Mengacu pada data pengujian tanpa RUS.
}  # Menutup kelompok pemeriksaan.

for nama, df in kelompok_pemeriksaan.items():  # Memeriksa proporsi kelas setiap bagian.
    beda_prevalensi = abs(df['label'].mean() - prevalensi) * 100  # Mengukur selisih prevalensi dalam poin persentase.
    if beda_prevalensi > 2.0:  # Mengidentifikasi perbedaan kelas yang relatif besar.
        print(f'Periksa distribusi kelas subset {nama}: selisih {beda_prevalensi:.2f} poin persentase.')  # Menampilkan selisih yang perlu dilaporkan.


In [ ]:
fig, ax = kanvas(bawah=0.20)  # Membuat kanvas jumlah responden antarkelompok.
kelompok = ['Latih', 'Validasi', 'Pengujian']  # Menentukan nama tiga subset eksperimen.
jumlah_split = [len(latih), len(validasi), len(uji)]  # Menghitung responden sebelum RUS untuk setiap subset.
ax.bar(kelompok, jumlah_split, color=[WARNA_GELAP, WARNA_SEDANG, WARNA_MUDA], width=0.60)  # Menggambar ukuran subset.
ax.set_ylabel('Jumlah responden')  # Menjelaskan jumlah data pada sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Menampilkan titik sebagai pemisah ribuan.
simpan_gambar(fig, '05_jumlah_responden_per_subset')  # Menyimpan visualisasi hasil pembagian.


In [ ]:
fig, ax = kanvas(bawah=0.20)  # Membuat kanvas jumlah responden antarkelompok.
kelompok = ['Latih', 'Validasi', 'Pengujian']  # Menentukan nama tiga subset eksperimen.
jumlah_split = [len(latih), len(validasi), len(uji)]  # Menghitung responden sebelum RUS untuk setiap subset.

# Menggambar ukuran subset dan simpan objek batangnya
bars = ax.bar(kelompok, jumlah_split, color=[WARNA_GELAP, WARNA_SEDANG, WARNA_MUDA], width=0.60)

# Membuat format teks dengan titik ribuan secara manual
labels_split = [f'{int(val):,}'.replace(',', '.') for val in jumlah_split]

# Menambahkan label nilai di atas masing-masing batang
ax.bar_label(bars, labels=labels_split, padding=4, fontsize=9, fontweight='bold')

ax.set_ylabel('Jumlah responden')  # Menjelaskan jumlah data pada sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Menampilkan titik sebagai pemisah ribuan.

# Menyesuaikan batas sumbu Y secara otomatis agar ada ruang untuk teks di atas batang tertinggi
max_split = max(jumlah_split)
ax.set_ylim(0, max_split * 1.20)

simpan_gambar(fig, '05_jumlah_responden_per_subset')  # Menyimpan visualisasi hasil pembagian.

In [ ]:
tabel_rus = ukuran[['No CVD History', 'CVD History']].copy()  # Mengambil angka kedua kelas pada empat kelompok data.
posisi = np.arange(len(tabel_rus))  # Menentukan posisi empat kelompok dalam grafik.
lebar = 0.35  # Menentukan lebar batang untuk tiap kelas.

fig, ax = kanvas(bawah=0.26)  # Membuat kanvas dengan ruang label kelompok panjang.
ax.bar(posisi - lebar / 2, tabel_rus['No CVD History'], width=lebar, color=WARNA_GELAP, label='No CVD History')  # Menggambar jumlah kelas mayoritas.
ax.bar(posisi + lebar / 2, tabel_rus['CVD History'], width=lebar, color=WARNA_MUDA, label='CVD History')  # Menggambar jumlah kelas minoritas.
ax.set_xticks(posisi, ['Latih sebelum RUS', 'Latih setelah RUS', 'Validasi', 'Pengujian'], rotation=12, ha='right')  # Menamai empat kelompok.
ax.set_ylabel('Jumlah responden')  # Menetapkan arti sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Memformat jumlah dengan titik ribuan.
ax.legend(frameon=False)  # Menampilkan nama dua kelas.
simpan_gambar(fig, '06_distribusi_kelas_dan_rus')  # Menampilkan hasil RUS dan menyimpannya.


In [ ]:
tabel_rus = ukuran[['No CVD History', 'CVD History']].copy()  # Mengambil angka kedua kelas pada empat kelompok data.
posisi = np.arange(len(tabel_rus))  # Menentukan posisi empat kelompok dalam grafik.
lebar = 0.35  # Menentukan lebar batang untuk tiap kelas.

fig, ax = kanvas(bawah=0.26)  # Membuat kanvas dengan ruang label kelompok panjang.

# Menggambar jumlah kelas mayoritas dan minoritas, lalu simpan objek batangnya
bars_no_cvd = ax.bar(posisi - lebar / 2, tabel_rus['No CVD History'], width=lebar, color=WARNA_GELAP, label='No CVD History')
bars_cvd = ax.bar(posisi + lebar / 2, tabel_rus['CVD History'], width=lebar, color=WARNA_MUDA, label='CVD History')

# Membuat format teks dengan titik ribuan secara manual untuk masing-masing kelas
labels_no_cvd = [f'{int(val):,}'.replace(',', '.') for val in tabel_rus['No CVD History']]
labels_cvd = [f'{int(val):,}'.replace(',', '.') for val in tabel_rus['CVD History']]

# Menambahkan label nilai jumlah responden di atas masing-masing batang menggunakan parameter labels
ax.bar_label(bars_no_cvd, labels=labels_no_cvd, padding=3, fontsize=8, rotation=0)
ax.bar_label(bars_cvd, labels=labels_cvd, padding=3, fontsize=8, rotation=0)

ax.set_xticks(posisi, ['Latih sebelum RUS', 'Latih setelah RUS', 'Validasi', 'Pengujian'], rotation=12, ha='right')  # Menamai empat kelompok.
ax.set_ylabel('Jumlah responden')  # Menetapkan arti sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Memformat jumlah dengan titik ribuan.

# Menyesuaikan batas sumbu Y secara otomatis agar ada ruang untuk teks di atas batang tertinggi
max_responden = tabel_rus.max().max()
ax.set_ylim(0, max_responden * 1.18)

ax.legend(frameon=False)  # Menampilkan nama dua kelas.
simpan_gambar(fig, '06_distribusi_kelas_dan_rus')  # Menampilkan hasil RUS dan menyimpannya.

In [ ]:
EN = {  # Mendefinisikan terjemahan kategori ke narasi bahasa Inggris.
    '_AGEG5YR': {  # Menentukan deskripsi setiap kelompok umur.
        1: '18-24',  # Kelompok umur 18 sampai 24 tahun.
        2: '25-29',  # Kelompok umur 25 sampai 29 tahun.
        3: '30-34',  # Kelompok umur 30 sampai 34 tahun.
        4: '35-39',  # Kelompok umur 35 sampai 39 tahun.
        5: '40-44',  # Kelompok umur 40 sampai 44 tahun.
        6: '45-49',  # Kelompok umur 45 sampai 49 tahun.
        7: '50-54',  # Kelompok umur 50 sampai 54 tahun.
        8: '55-59',  # Kelompok umur 55 sampai 59 tahun.
        9: '60-64',  # Kelompok umur 60 sampai 64 tahun.
        10: '65-69',  # Kelompok umur 65 sampai 69 tahun.
        11: '70-74',  # Kelompok umur 70 sampai 74 tahun.
        12: '75-79',  # Kelompok umur 75 sampai 79 tahun.
        13: '80+',  # Kelompok umur 80 tahun atau lebih.
    },  # Mengakhiri pemetaan kelompok umur.
    '_SEX': {  # Menentukan kode jenis kelamin.
        1: 'male',  # Mengartikan kode 1 sebagai laki-laki.
        2: 'female',  # Mengartikan kode 2 sebagai perempuan.
    },  # Mengakhiri pemetaan jenis kelamin.
    '_BMI5CAT': {  # Menentukan kategori indeks massa tubuh.
        1: 'underweight',  # Mengartikan BMI kurang.
        2: 'normal weight',  # Mengartikan BMI normal.
        3: 'overweight',  # Mengartikan BMI berlebih.
        4: 'obese',  # Mengartikan obesitas.
    },  # Mengakhiri kategori BMI.
    'GENHLTH': {  # Menentukan kategori kondisi kesehatan umum.
        1: 'excellent',  # Mengartikan kesehatan sangat baik.
        2: 'very good',  # Mengartikan kesehatan baik sekali.
        3: 'good',  # Mengartikan kesehatan baik.
        4: 'fair',  # Mengartikan kesehatan cukup.
        5: 'poor',  # Mengartikan kesehatan buruk.
    },  # Mengakhiri kategori kondisi kesehatan umum.
    'DIABETE4': {  # Menentukan kategori status diabetes.
        1: 'diagnosed diabetes',  # Mengartikan pernah didiagnosis diabetes.
        2: 'diabetes during pregnancy only',  # Mengartikan diabetes hanya selama kehamilan.
        3: 'no diabetes',  # Mengartikan tidak melaporkan diabetes.
        4: 'prediabetes',  # Mengartikan pradiabetes.
    },  # Mengakhiri kategori status diabetes.
    'CHCKDNY2': {  # Menentukan riwayat penyakit ginjal.
        1: 'yes',  # Mengartikan ada riwayat ginjal.
        2: 'no',  # Mengartikan tidak melaporkan riwayat ginjal.
    },  # Mengakhiri kategori penyakit ginjal.
    'CVDSTRK3': {  # Menentukan riwayat stroke.
        1: 'yes',  # Mengartikan ada riwayat stroke.
        2: 'no',  # Mengartikan tidak melaporkan riwayat stroke.
    },  # Mengakhiri kategori stroke.
    'CHCCOPD3': {  # Menentukan riwayat COPD.
        1: 'yes',  # Mengartikan ada riwayat COPD.
        2: 'no',  # Mengartikan tidak melaporkan riwayat COPD.
    },  # Mengakhiri kategori COPD.
    '_RFSMOK3': {  # Menentukan indikator merokok saat ini.
        1: 'no',  # Mengartikan tidak merokok saat ini.
        2: 'yes',  # Mengartikan merokok saat ini.
    },  # Mengakhiri status merokok.
    '_TOTINDA': {  # Menentukan aktivitas fisik.
        1: 'yes',  # Mengartikan memenuhi indikator aktivitas fisik.
        2: 'no',  # Mengartikan tidak memenuhi indikator aktivitas fisik.
    },  # Mengakhiri indikator aktivitas fisik.
    '_RFBING6': {  # Menentukan indikator binge drinking.
        1: 'no',  # Mengartikan tidak memiliki indikator binge drinking.
        2: 'yes',  # Mengartikan memiliki indikator binge drinking.
    },  # Mengakhiri indikator binge drinking.
}  # Mengakhiri seluruh pemetaan kategori.

NAMA = [  # Menentukan nama atribut yang digunakan pada templat teks.
    'age group',  # Menggambarkan kelompok usia.
    'sex',  # Menggambarkan jenis kelamin.
    'BMI category',  # Menggambarkan indeks massa tubuh.
    'general health',  # Menggambarkan penilaian kesehatan umum.
    'diabetes status',  # Menggambarkan kondisi diabetes.
    'kidney disease history',  # Menggambarkan riwayat penyakit ginjal.
    'stroke history',  # Menggambarkan riwayat stroke.
    'COPD history',  # Menggambarkan riwayat COPD.
    'current smoking',  # Menggambarkan indikator merokok.
    'physical activity',  # Menggambarkan indikator aktivitas fisik.
    'binge drinking',  # Menggambarkan indikator konsumsi alkohol berlebihan.
]  # Mengakhiri daftar nama atribut.

def jadikan_teks(baris):  # Mengubah satu rekaman responden menjadi satu teks bahasa Inggris.
    bagian_teks = []  # Menyimpan deskripsi atribut dalam urutan yang konsisten.
    for fitur, nama in zip(FITUR, NAMA):  # Membaca setiap atribut secara berurutan.
        nilai = baris[fitur]  # Mengambil nilai kategori dari responden.
        keterangan = EN[fitur][nilai]  # Mengubah nilai menjadi keterangan bahasa Inggris.
        bagian_teks.append(f'{nama}: {keterangan}')  # Menambahkan deskripsi atribut pada narasi.
    narasi = '; '.join(bagian_teks)  # Menggabungkan keterangan menggunakan tanda titik koma.
    return 'The respondent has the following health profile: ' + narasi + '.'  # Mengembalikan teks tanpa label target.

for df in (latih_rus, validasi, uji):  # Menerapkan templat yang sama pada seluruh subset.
    df['text'] = df.apply(jadikan_teks, axis=1)  # Menyusun narasi tiap responden.

latih_rus[['text', 'label']].to_csv(OUT / 'teks_latih.csv', index=False)  # Menyimpan narasi data pelatihan.
validasi[['text', 'label']].to_csv(OUT / 'teks_validasi.csv', index=False)  # Menyimpan narasi data validasi.
uji[['text', 'label']].to_csv(OUT / 'teks_uji.csv', index=False)  # Menyimpan narasi data pengujian.

print('Contoh narasi bahasa Inggris:')  # Memberi judul contoh representasi.
print(latih_rus.iloc[0]['text'])  # Menampilkan narasi aktual dari salah satu responden latih.


In [ ]:
from transformers import AutoTokenizer  # Mengakses tokenizer bawaan TinyLlama.
from transformers import AutoModelForSequenceClassification  # Mengakses arsitektur klasifikasi TinyLlama.
from transformers import BitsAndBytesConfig  # Mengatur kuantisasi bobot model 4-bit.
from peft import LoraConfig  # Menentukan konfigurasi adapter LoRA.
from peft import get_peft_model  # Memasang adapter LoRA pada model dasar.
from peft import prepare_model_for_kbit_training  # Menyiapkan model terkuantisasi untuk training.
from torch.utils.data import TensorDataset  # Menyatukan token dan label ke dataset tensor.
from torch.utils.data import DataLoader  # Menyajikan tensor dalam beberapa batch.
from tqdm.auto import tqdm  # Menampilkan progres pelatihan pada Colab.

tokenizer = AutoTokenizer.from_pretrained(MODEL)  # Memuat tokenizer sesuai checkpoint TinyLlama.
tokenizer.pad_token = tokenizer.eos_token  # Menggunakan token akhir sebagai token padding.
panjang_max = 0  # Menyediakan variabel untuk panjang teks terpanjang.

for df in (latih_rus, validasi, uji):  # Memeriksa seluruh narasi dari tiga subset.
    for posisi in range(0, len(df), 2048):  # Memeriksa teks dalam potongan agar penggunaan RAM terkendali.
        teks_batch = df['text'].iloc[posisi:posisi + 2048].tolist()  # Mengambil maksimal 2.048 narasi.
        token_batch = tokenizer(  # Melakukan audit token tanpa pemotongan teks.
            teks_batch,  # Memasukkan daftar narasi bahasa Inggris.
            add_special_tokens=True,  # Menggunakan token khusus sesuai tokenizer.
            truncation=False,  # Mencegah penghilangan fitur selama audit.
        )  # Mengakhiri tokenisasi untuk pemeriksaan panjang.
        panjang_batch = max(map(len, token_batch['input_ids']))  # Mencari panjang terbesar dalam potongan.
        panjang_max = max(panjang_max, panjang_batch)  # Memperbarui panjang token terbesar.

print('Panjang token terpanjang:', panjang_max)  # Menampilkan hasil audit panjang teks.
assert panjang_max <= MAX_TOKEN, f'Teks {panjang_max} token melebihi MAX_TOKEN={MAX_TOKEN}.'  # Menghentikan proses apabila ada fitur yang akan terpotong.
PAD_LEN = panjang_max  # Menggunakan panjang aktual untuk mengurangi padding yang tidak diperlukan.

def buat_loader(df, batch_size, acak=False):  # Membentuk DataLoader dari narasi dan label.
    token = tokenizer(  # Mengubah seluruh teks subset menjadi tensor token.
        df['text'].tolist(),  # Mengambil semua narasi dalam subset.
        padding='max_length',  # Menambahkan padding sampai panjang aktual terpanjang.
        max_length=PAD_LEN,  # Memakai panjang yang telah diperiksa sebelumnya.
        truncation=False,  # Tidak memotong informasi sebelas fitur.
        return_tensors='pt',  # Menghasilkan tensor PyTorch.
    )  # Mengakhiri proses tokenisasi subset.
    label = torch.tensor(df['label'].to_numpy(), dtype=torch.long)  # Mengubah label menjadi tensor integer.
    tensor = TensorDataset(token['input_ids'], token['attention_mask'], label)  # Menyatukan token, mask, dan label.
    loader = DataLoader(  # Menyusun data menjadi batch untuk GPU.
        tensor,  # Menggunakan dataset tensor hasil tokenisasi.
        batch_size=batch_size,  # Menentukan jumlah sampel setiap batch.
        shuffle=acak,  # Mengacak data hanya ketika diminta.
        pin_memory=True,  # Mempercepat transfer tensor ke GPU.
        num_workers=0,  # Menghindari masalah multiprocessing pada Colab.
    )  # Mengakhiri konfigurasi DataLoader.
    return loader  # Mengembalikan loader yang siap digunakan.

loader_latih = buat_loader(latih_rus, BATCH, acak=True)  # Menyiapkan batch data latih yang telah diseimbangkan.
loader_val = buat_loader(validasi, BATCH_EVAL)  # Menyiapkan batch validasi tanpa RUS.
loader_uji = buat_loader(uji, BATCH_EVAL)  # Menyiapkan batch pengujian tanpa RUS.

print('Batch pelatihan:', len(loader_latih))  # Menampilkan jumlah iterasi pelatihan setiap epoch.
print('Batch validasi:', len(loader_val))  # Menampilkan jumlah iterasi validasi.
print('Batch pengujian:', len(loader_uji))  # Menampilkan jumlah iterasi pengujian.


In [ ]:
panjang_latih = loader_latih.dataset.tensors[1].sum(dim=1).numpy()  # Menghitung token aktif pada setiap narasi data latih.
panjang_validasi = loader_val.dataset.tensors[1].sum(dim=1).numpy()  # Menghitung token aktif pada setiap narasi validasi.
panjang_uji = loader_uji.dataset.tensors[1].sum(dim=1).numpy()  # Menghitung token aktif pada setiap narasi pengujian.
ringkasan_token = pd.DataFrame({  # Merangkum panjang token masing-masing bagian data.
    'Subset': ['Pelatihan setelah RUS', 'Validasi', 'Pengujian'],  # Mengidentifikasi tiga kelompok masukan.
    'Rata-rata token': [np.mean(panjang_latih), np.mean(panjang_validasi), np.mean(panjang_uji)],  # Mencatat rerata panjang token.
    'Panjang maksimum': [np.max(panjang_latih), np.max(panjang_validasi), np.max(panjang_uji)],  # Mencatat batas tertinggi setiap kelompok.
})  # Mengakhiri tabel panjang masukan.
ringkasan_token.to_csv(OUT / 'ringkasan_panjang_token.csv', index=False)  # Menyimpan audit panjang token untuk Bab IV.
display(ringkasan_token.round(2))  # Menampilkan ringkasan penggunaan token.

fig, ax = kanvas()  # Menyiapkan kanvas histogram panjang token.
batas_bin = np.arange(0, MAX_TOKEN + 3, 3)  # Membagi panjang token menjadi interval tiga token.
ax.hist(panjang_latih, bins=batas_bin, histtype='step', linewidth=2, color=WARNA_GELAP, label='Pelatihan setelah RUS')  # Menggambar distribusi token latih.
ax.hist(panjang_validasi, bins=batas_bin, histtype='step', linewidth=2, color=WARNA_SEDANG, label='Validasi')  # Menggambar distribusi token validasi.
ax.hist(panjang_uji, bins=batas_bin, histtype='step', linewidth=2, color=WARNA_MUDA, label='Pengujian')  # Menggambar distribusi token pengujian.
ax.axvline(MAX_TOKEN, linestyle='--', color=WARNA_PUCAT, linewidth=1, label='Batas maksimum 128 token')  # Menandai batas rancangan.
ax.set_xlabel('Panjang urutan masukan (token)')  # Menamai panjang urutan pada sumbu horizontal.
ax.set_ylabel('Jumlah responden')  # Menamai banyaknya responden pada sumbu vertikal.
ax.yaxis.set_major_formatter(FuncFormatter(format_ribuan))  # Memformat sumbu jumlah responden.
ax.legend(frameon=False)  # Menjelaskan tiga subset dan batas token.
simpan_gambar(fig, '07_distribusi_panjang_token')  # Menyimpan grafik segera setelah tokenisasi selesai.


In [ ]:
bnb = BitsAndBytesConfig(  # Menentukan pengaturan kuantisasi model dasar.
    load_in_4bit=True,  # Memuat bobot model dalam representasi 4-bit.
    bnb_4bit_quant_type='nf4',  # Menggunakan kuantisasi NormalFloat4.
    bnb_4bit_compute_dtype=torch.float16,  # Menggunakan FP16 untuk perhitungan pada GPU.
    bnb_4bit_use_double_quant=True,  # Mengaktifkan kuantisasi bertingkat untuk menghemat memori.
)  # Mengakhiri konfigurasi kuantisasi.

def buat_model(metode):  # Menyiapkan TinyLlama sesuai pendekatan eksperimen.
    torch.manual_seed(SEED)  # Menyamakan inisialisasi acak untuk setiap pendekatan.
    torch.cuda.manual_seed_all(SEED)  # Menyamakan inisialisasi acak pada GPU.
    model = AutoModelForSequenceClassification.from_pretrained(  # Memuat TinyLlama sebagai model klasifikasi.
        MODEL,  # Menggunakan checkpoint TinyLlama yang sama.
        num_labels=2,  # Menetapkan dua kelas keluaran.
        quantization_config=bnb,  # Menggunakan kuantisasi NF4 4-bit.
        device_map={'': 0},  # Menempatkan model pada GPU pertama.
    )  # Mengakhiri pemuatan model.

    model.config.pad_token_id = tokenizer.pad_token_id  # Menyamakan ID padding dengan tokenizer.
    model.config.use_cache = False  # Mematikan cache agar sesuai untuk pelatihan.
    model.score = model.score.float()  # Menyiapkan classification head dalam FP32.

    if metode == 'frozen':  # Memilih pendekatan frozen backbone.
        for parameter in model.parameters():  # Mengunjungi seluruh bobot model.
            parameter.requires_grad_(False)  # Membekukan semua bobot dasar.
        for parameter in model.score.parameters():  # Mengunjungi lapisan klasifikasi.
            parameter.requires_grad_(True)  # Mengaktifkan pelatihan hanya pada classification head.
    elif metode == 'qlora':  # Memilih pendekatan fine-tuning menggunakan QLoRA.
        model = prepare_model_for_kbit_training(  # Menyiapkan TinyLlama terkuantisasi untuk gradien LoRA.
            model,  # Menyediakan model yang telah dimuat.
            gradient_checkpointing_kwargs={'use_reentrant': False},  # Mengatur checkpointing agar kompatibel dengan PyTorch.
        )  # Mengakhiri persiapan model k-bit.
        cfg = LoraConfig(  # Membuat konfigurasi adapter LoRA.
            r=64,  # Menetapkan rank LoRA.
            lora_alpha=32,  # Menetapkan skala alpha LoRA.
            lora_dropout=0.05,  # Menetapkan dropout untuk adapter.
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # Menentukan modul perhatian yang diadaptasi.
            task_type='SEQ_CLS',  # Menetapkan tugas klasifikasi urutan.
            modules_to_save=['score'],  # Mengikutsertakan classification head dalam proses pelatihan.
        )  # Mengakhiri konfigurasi LoRA.
        model = get_peft_model(model, cfg)  # Memasang adapter LoRA pada TinyLlama.
    else:  # Menangani nama metode yang tidak dikenali.
        raise ValueError('Metode harus frozen atau qlora.')  # Menghentikan pemanggilan dengan nama yang salah.

    for parameter in model.parameters():  # Memeriksa tipe data seluruh bobot.
        if parameter.requires_grad and parameter.dtype != torch.float32:  # Memilih bobot latih yang belum FP32.
            parameter.data = parameter.data.float()  # Mengubah hanya bobot yang dilatih menjadi FP32.

    parameter_latih = [p for p in model.parameters() if p.requires_grad]  # Mengumpulkan bobot yang akan diperbarui.
    assert len(parameter_latih) > 0, 'Tidak ada parameter yang dilatih.'  # Memastikan model memiliki parameter trainable.
    assert all(p.dtype == torch.float32 for p in parameter_latih), 'Masih ada parameter FP16 yang dilatih.'  # Mencegah error GradScaler.

    jumlah_parameter = sum(p.numel() for p in parameter_latih)  # Menghitung jumlah parameter trainable.
    print('Metode:', metode)  # Menampilkan pendekatan yang dipilih.
    print('Jumlah parameter dilatih:', jumlah_parameter)  # Menampilkan beban parameter pelatihan.
    return model  # Mengembalikan model siap dilatih.


In [ ]:
def evaluasi(model, loader, hitung_loss=True):  # Menghitung prediksi dan loss tanpa mengubah bobot.
    model.eval()  # Mengaktifkan mode evaluasi agar dropout tidak bekerja.
    aktual = []  # Menampung label kelas sebenarnya.
    prediksi = []  # Menampung label kelas hasil prediksi.
    total_loss = 0.0  # Menampung jumlah loss tertimbang ukuran batch.
    total = 0  # Menghitung banyaknya sampel yang dievaluasi.

    with torch.inference_mode():  # Menonaktifkan perhitungan gradien pada evaluasi.
        with torch.autocast('cuda', dtype=torch.float16):  # Menghemat memori melalui mixed precision.
            for x, mask, y in tqdm(loader, desc='Validasi / pengujian', unit='batch', leave=False):  # Menampilkan progres setiap batch evaluasi.
                x = x.cuda(non_blocking=True)  # Memindahkan token masukan ke GPU.
                mask = mask.cuda(non_blocking=True)  # Memindahkan attention mask ke GPU.
                y = y.cuda(non_blocking=True)  # Memindahkan label ke GPU.
                label_loss = y if hitung_loss else None  # Mengaktifkan loss hanya bila diperlukan.
                keluaran = model(input_ids=x, attention_mask=mask, labels=label_loss)  # Menjalankan prediksi model.
                if hitung_loss:  # Memeriksa apakah loss perlu dihitung.
                    total_loss += keluaran.loss.item() * len(y)  # Menambahkan loss batch dengan bobot jumlah sampel.
                total += len(y)  # Menambahkan jumlah responden yang dievaluasi.
                aktual.extend(y.cpu().tolist())  # Menyimpan label aktual ke memori CPU.
                prediksi.extend(keluaran.logits.argmax(dim=-1).cpu().tolist())  # Menyimpan kelas hasil prediksi.

    rerata_loss = total_loss / total if hitung_loss else None  # Menghitung rata-rata loss seluruh sampel.
    return np.array(aktual), np.array(prediksi), rerata_loss  # Mengembalikan label aktual, prediksi, dan loss.

def metrik(y_aktual, y_prediksi):  # Menghitung metrik klasifikasi utama dan kelas CVD.
    presisi, recall, f1_cvd, _ = precision_recall_fscore_support(  # Menghitung metrik hanya untuk kelas positif.
        y_aktual,  # Memasukkan label sebenarnya.
        y_prediksi,  # Memasukkan label prediksi.
        labels=[1],  # Memilih kelas CVD History sebagai kelas positif.
        zero_division=0,  # Menghindari pembagian nol saat prediksi suatu kelas kosong.
    )  # Mengakhiri perhitungan metrik kelas positif.
    hasil = {  # Menyusun metrik penelitian dalam satu objek.
        'akurasi': float(accuracy_score(y_aktual, y_prediksi)),  # Menghitung proporsi prediksi benar.
        'macro_f1': float(f1_score(y_aktual, y_prediksi, average='macro', zero_division=0)),  # Menghitung rata-rata F1 dua kelas.
        'presisi_cvd': float(presisi[0]),  # Mengambil presisi kelas CVD History.
        'recall_cvd': float(recall[0]),  # Mengambil recall kelas CVD History.
        'f1_cvd': float(f1_cvd[0]),  # Mengambil F1-score kelas CVD History.
    }  # Mengakhiri daftar metrik.
    return hasil  # Mengembalikan hasil evaluasi model.

hash_sampel = pd.util.hash_pandas_object(  # Menghitung identitas setiap baris sampel beserta labelnya.
    sampel[FITUR + ['label']],  # Memakai fitur dan label sebagai isi data acuan.
    index=False,  # Mengabaikan indeks DataFrame saat membangun hash.
)  # Menutup perhitungan hash sampel.

penanda_split = pd.Series(index=sampel.index, dtype='string')  # Menyiapkan penampung identitas subset setiap responden.
penanda_split.loc[latih.index] = 'latih'  # Menandai responden yang masuk data latih.
penanda_split.loc[validasi.index] = 'validasi'  # Menandai responden yang masuk data validasi.
penanda_split.loc[uji.index] = 'uji'  # Menandai responden yang masuk data pengujian.
assert penanda_split.notna().all(), 'Ada responden tanpa penanda subset.'  # Memastikan seluruh baris mendapat satu subset.

kunci_split = hashlib.sha256(  # Membuat identitas pembagian yang benar-benar dipakai.
    penanda_split.astype(str).str.cat(sep='|').encode('utf-8')  # Menggabungkan penanda subset sesuai urutan sampel.
).hexdigest()[:16]  # Mengambil ringkasan identitas pembagian.

FINGERPRINT = hashlib.sha256(hash_sampel.values.tobytes()).hexdigest()[:16]  # Membuat identitas sampel penelitian.

identitas_eksperimen = {  # Mengumpulkan unsur eksperimen yang tidak boleh berubah saat melanjutkan checkpoint.
    'dataset': FINGERPRINT,  # Menandai data terpilih dan labelnya.
    'split': kunci_split,  # Menandai susunan data latih, validasi, dan pengujian.
    'model': MODEL,  # Menandai checkpoint TinyLlama yang digunakan.
    'seed': SEED,  # Menandai seed pengacakan.
    'n_sampel': N_SAMPEL,  # Menandai ukuran sampel penelitian.
    'epoch': EPOCHS,  # Menandai jumlah maksimum epoch.
    'batch': BATCH,  # Menandai jumlah contoh per batch latih.
    'batch_eval': BATCH_EVAL,  # Menandai jumlah contoh per batch validasi.
    'max_token': MAX_TOKEN,  # Menandai batas panjang token.
    'pemilihan_split': JUMLAH_KANDIDAT,  # Menandai banyak kandidat pembagian profil.
    'sampling': 'stratified_random',  # Menandai teknik pengambilan sampel.
    'split_mode': 'best_of_group_shuffle',  # Menandai aturan pembagian antarprofil.
    'rus': 'train_only_1_to_1',  # Menandai aturan ketidakseimbangan kelas.
    'lora_r': 64,  # Menandai rank adapter LoRA.
    'lora_alpha': 32,  # Menandai alpha adapter LoRA.
    'lora_dropout': 0.05,  # Menandai dropout adapter LoRA.
    'lora_target': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # Menandai modul yang diadaptasi.
    'lr': 1e-4,  # Menandai learning rate.
    'weight_decay': 0.01,  # Menandai regularisasi optimizer.
}  # Menutup pengumpulan identitas eksperimen.

SIGNATURE = hashlib.sha256(  # Membuat penanda yang menolak checkpoint dari rancangan berbeda.
    json.dumps(identitas_eksperimen, sort_keys=True).encode('utf-8')  # Mengubah konfigurasi menjadi teks tetap.
).hexdigest()  # Menghasilkan signature untuk pemeriksaan checkpoint.

def simpan_checkpoint(isi, tujuan):  # Menyimpan model atau status pelatihan ke Drive.
    sementara = Path('/content') / tujuan.name  # Menyiapkan lokasi penulisan sementara di Colab.
    torch.save(isi, sementara)  # Menulis berkas sampai selesai di disk lokal.
    shutil.copy2(sementara, tujuan)  # Menyalin hasil penulisan lengkap ke Drive.
    sementara.unlink(missing_ok=True)  # Menghapus berkas sementara setelah disalin.

def ambil_bobot(model):  # Mengambil hanya parameter yang dilatih.
    bobot = {}  # Menyediakan penampung pasangan nama dan bobot.
    for nama, parameter in model.named_parameters():  # Menelusuri parameter model.
        if parameter.requires_grad:  # Memilih bobot yang diperbarui selama pelatihan.
            bobot[nama] = parameter.detach().cpu().clone()  # Menyalin bobot dari GPU ke CPU.
    return bobot  # Mengembalikan kumpulan bobot trainable.

def pulihkan_bobot(model, bobot):  # Memasang kembali bobot model dari checkpoint.
    parameter_model = dict(model.named_parameters())  # Membuat daftar parameter yang dapat diakses berdasarkan nama.
    nama_aktif = {nama for nama, p in parameter_model.items() if p.requires_grad}  # Mengambil nama seluruh parameter trainable.
    assert set(bobot) == nama_aktif, 'Nama parameter dalam checkpoint berbeda.'  # Menolak checkpoint dari konfigurasi lain.
    with torch.no_grad():  # Mencegah pemulihan bobot tercatat sebagai operasi pelatihan.
        for nama, nilai in bobot.items():  # Menelusuri bobot yang akan dipulihkan.
            tujuan = parameter_model[nama]  # Mengambil parameter dengan nama yang cocok.
            tujuan.copy_(nilai.to(tujuan.device))  # Memindahkan bobot ke perangkat dan parameter yang benar.


In [ ]:
def eksperimen(metode):  # Melatih satu pendekatan dan mengevaluasinya pada data uji.
    result_file = OUT / f'{metode}_hasil_test.json'  # Menentukan file metrik akhir.
    ckpt_file = OUT / f'{metode}_checkpoint_lanjut.pt'  # Menentukan file pemulihan dari epoch sebelumnya.
    best_file = OUT / f'{metode}_bobot_terbaik.pt'  # Menentukan file bobot terbaik dari validasi.

    if result_file.exists():  # Memeriksa apakah pengujian metode ini pernah selesai.
        tersimpan = json.loads(result_file.read_text())  # Membaca hasil pengujian yang tersimpan.
        assert tersimpan['signature'] == SIGNATURE, 'Hasil lama berbeda konfigurasi.'  # Memastikan hasil berasal dari eksperimen yang sama.
        print(metode, 'sudah selesai; membaca hasil tersimpan.')  # Menampilkan status hasil yang tersedia.
        return tersimpan['metrik']  # Mengembalikan skor tanpa mengulangi pelatihan.

    model = buat_model(metode)  # Membuat model frozen backbone atau QLoRA.
    params = [p for p in model.parameters() if p.requires_grad]  # Memilih hanya bobot yang boleh diperbarui.
    opt = torch.optim.AdamW(  # Membuat optimizer untuk parameter yang dilatih.
        params,  # Memberikan parameter trainable ke optimizer.
        lr=1e-4,  # Menetapkan learning rate awal.
        weight_decay=0.01,  # Mengatur regularisasi optimizer.
    )  # Mengakhiri pengaturan optimizer.
    scaler = torch.amp.GradScaler('cuda')  # Mengaktifkan penskalaan gradien untuk FP16.

    skor_terbaik = -1.0  # Menyimpan nilai macro F1 validasi terbaik.
    riwayat = []  # Menampung loss dan metrik setiap epoch.
    epoch_mulai = 1  # Menetapkan epoch awal jika belum ada checkpoint.

    if ckpt_file.exists():  # Memeriksa status pelatihan yang disimpan sebelumnya.
        ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=True)  # Membaca berkas checkpoint di CPU.
        assert ckpt['signature'] == SIGNATURE, 'Checkpoint berbeda konfigurasi.'  # Memastikan konfigurasi sama.
        pulihkan_bobot(model, ckpt['bobot'])  # Mengembalikan parameter trainable terakhir.
        opt.load_state_dict(ckpt['optimizer'])  # Mengembalikan status optimizer dari epoch terakhir.
        scaler.load_state_dict(ckpt['scaler'])  # Mengembalikan status gradien FP16.
        skor_terbaik = ckpt['best']  # Mengembalikan skor validasi terbaik yang pernah tercatat.
        riwayat = ckpt['riwayat']  # Mengembalikan tabel hasil epoch sebelumnya.
        epoch_mulai = ckpt['epoch'] + 1  # Melanjutkan dari epoch berikutnya.
        print('Melanjutkan', metode, 'dari epoch', epoch_mulai)  # Menampilkan posisi pelatihan yang dipulihkan.

    print('Batch latih per epoch:', len(loader_latih))  # Menampilkan banyak batch pelatihan.
    print('Batch validasi per epoch:', len(loader_val))  # Menampilkan banyak batch validasi.

    for epoch in range(epoch_mulai, EPOCHS + 1):  # Melatih hingga jumlah maksimum epoch.
        torch.manual_seed(SEED + epoch)  # Mengendalikan pengacakan pada epoch sekarang.
        torch.cuda.manual_seed_all(SEED + epoch)  # Mengendalikan pengacakan GPU pada epoch sekarang.
        mulai_epoch = time.perf_counter()  # Menandai waktu mulai satu epoch.
        torch.cuda.reset_peak_memory_stats()  # Mengulang penghitung pemakaian puncak memori GPU.
        model.train()  # Mengaktifkan dropout dan perilaku pelatihan.
        total_loss = 0.0  # Menyediakan akumulator training loss.
        total_data = 0  # Menyediakan penghitung responden yang dipakai selama epoch.

        progres = tqdm(loader_latih, desc=f'{metode} epoch {epoch}/{EPOCHS}', unit='batch')  # Menampilkan progres setiap batch.
        for x, mask, y in progres:  # Memproses seluruh batch latih pada epoch sekarang.
            x = x.cuda(non_blocking=True)  # Memindahkan token input ke GPU.
            mask = mask.cuda(non_blocking=True)  # Memindahkan attention mask ke GPU.
            y = y.cuda(non_blocking=True)  # Memindahkan label latih ke GPU.
            opt.zero_grad(set_to_none=True)  # Menghapus gradien batch sebelumnya.
            with torch.autocast('cuda', dtype=torch.float16):  # Menggunakan mixed precision selama forward pass.
                loss = model(input_ids=x, attention_mask=mask, labels=y).loss  # Menghitung cross-entropy loss data latih.
            scaler.scale(loss).backward()  # Menghitung gradien setelah loss diskalakan.
            scaler.step(opt)  # Memperbarui bobot yang dilatih melalui optimizer.
            scaler.update()  # Menyesuaikan skala gradien untuk batch berikutnya.
            total_loss += loss.item() * len(y)  # Mengakumulasi loss berbobot banyaknya responden.
            total_data += len(y)  # Mengakumulasi banyaknya responden pada epoch ini.

        durasi_latih = (time.perf_counter() - mulai_epoch) / 60  # Mengukur menit pelatihan pada epoch sekarang.
        mulai_validasi = time.perf_counter()  # Menandai awal perhitungan metrik validasi.
        train_loss = total_loss / total_data  # Menghitung rerata training loss sepanjang epoch.
        y_val, pred_val, val_loss = evaluasi(model, loader_val, hitung_loss=True)  # Menghitung prediksi dan validation loss pada data alami.
        nilai_val = metrik(y_val, pred_val)  # Menghitung metrik validasi.
        skor_val = nilai_val['macro_f1']  # Menggunakan macro F1 sebagai indikator pemilihan model.
        durasi_validasi = (time.perf_counter() - mulai_validasi) / 60  # Mengukur menit validasi.
        durasi_epoch = (time.perf_counter() - mulai_epoch) / 60  # Mengukur total menit satu epoch.
        gpu_puncak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Mengukur penggunaan memori GPU maksimum.

        catatan_epoch = {  # Membentuk catatan hasil satu epoch.
            'epoch': epoch,  # Menyimpan nomor epoch.
            'train_loss': float(train_loss),  # Menyimpan rerata training loss pada data RUS.
            'val_loss': float(val_loss),  # Menyimpan rerata validation loss pada data alami.
            'val_macro_f1': float(skor_val),  # Menyimpan macro F1 validasi.
            'val_akurasi': float(nilai_val['akurasi']),  # Menyimpan akurasi validasi.
            'durasi_latih_menit': float(durasi_latih),  # Mencatat waktu pelatihan per epoch.
            'durasi_validasi_menit': float(durasi_validasi),  # Mencatat waktu validasi per epoch.
            'durasi_total_menit': float(durasi_epoch),  # Mencatat waktu keseluruhan per epoch.
            'vram_puncak_gb': float(gpu_puncak_gb),  # Mencatat penggunaan memori maksimum selama epoch.
        }  # Mengakhiri catatan hasil satu epoch.

        riwayat.append(catatan_epoch)  # Menambahkan catatan ke riwayat pelatihan.
        pd.DataFrame(riwayat).to_csv(OUT / f'{metode}_loss_dan_metrik_per_epoch.csv', index=False)  # Menyimpan nilai loss dan metrik setiap epoch.
        print(  # Menampilkan nilai loss, validasi, dan waktu satu epoch.
            f'{metode} epoch {epoch}: train loss={train_loss:.4f}, '  # Menampilkan training loss.
            f'val loss={val_loss:.4f}, macro F1={skor_val:.4f}, '  # Menampilkan validation loss dan macro F1.
            f'durasi={durasi_epoch:.1f} menit'  # Menampilkan total waktu per epoch.
        )  # Menutup tampilan hasil epoch.

        if skor_val > skor_terbaik:  # Memeriksa apakah validasi sekarang lebih baik daripada sebelumnya.
            skor_terbaik = skor_val  # Memperbarui macro F1 terbaik.
            isi_terbaik = {  # Menyiapkan data bobot dari model terbaik.
                'signature': SIGNATURE,  # Mencatat identitas konfigurasi eksperimen.
                'bobot': ambil_bobot(model),  # Menyimpan parameter trainable model terbaik.
                'epoch': epoch,  # Mencatat epoch terbaik.
            }  # Mengakhiri data model terbaik.
            simpan_checkpoint(isi_terbaik, best_file)  # Menyimpan bobot terbaik ke Google Drive.

        isi_lanjut = {  # Menyiapkan data untuk melanjutkan pelatihan jika Colab terputus.
            'signature': SIGNATURE,  # Mencatat identitas konfigurasi.
            'epoch': epoch,  # Mencatat epoch yang telah selesai.
            'best': float(skor_terbaik),  # Mencatat skor validasi terbaik.
            'riwayat': riwayat,  # Mencatat riwayat seluruh epoch.
            'bobot': ambil_bobot(model),  # Mencatat bobot terakhir yang dilatih.
            'optimizer': opt.state_dict(),  # Mencatat momentum dan status optimizer.
            'scaler': scaler.state_dict(),  # Mencatat status mixed precision.
        }  # Mengakhiri data checkpoint lanjutan.
        simpan_checkpoint(isi_lanjut, ckpt_file)  # Menyimpan checkpoint setelah satu epoch selesai.

    terbaik = torch.load(best_file, map_location='cpu', weights_only=True)  # Memuat bobot berdasarkan validasi terbaik.
    assert terbaik['signature'] == SIGNATURE, 'Identitas bobot terbaik tidak sesuai.'  # Memastikan bobot sesuai konfigurasi.
    pulihkan_bobot(model, terbaik['bobot'])  # Menggunakan bobot dengan macro F1 validasi tertinggi.

    y_uji, pred_uji, _ = evaluasi(model, loader_uji, hitung_loss=False)  # Melakukan pengujian akhir pada data yang belum digunakan.
    hasil = metrik(y_uji, pred_uji)  # Menghitung metrik pengujian akhir.

    prediksi_uji = pd.DataFrame({'aktual': y_uji, 'prediksi': pred_uji})  # Membuat tabel label aktual dan prediksi.
    prediksi_uji.to_csv(OUT / f'{metode}_prediksi_uji.csv', index=False)  # Menyimpan prediksi responden pengujian.

    cm = confusion_matrix(y_uji, pred_uji, labels=[0, 1])  # Menghitung confusion matrix dua kelas.
    tabel_cm = pd.DataFrame(cm, index=['Aktual 0', 'Aktual 1'], columns=['Prediksi 0', 'Prediksi 1'])  # Memberi nama baris dan kolom matriks.
    tabel_cm.to_csv(OUT / f'{metode}_confusion_matrix.csv')  # Menyimpan confusion matrix.

    laporan = classification_report(  # Menyusun metrik lengkap untuk setiap kelas.
        y_uji,  # Memberikan label aktual.
        pred_uji,  # Memberikan hasil prediksi.
        labels=[0, 1],  # Menetapkan urutan kedua kelas.
        target_names=['No CVD History', 'CVD History'],  # Menampilkan nama kelas.
        output_dict=True,  # Menghasilkan laporan sebagai data terstruktur.
        zero_division=0,  # Menangani kelas tanpa prediksi.
    )  # Mengakhiri penyusunan laporan klasifikasi.

    pd.DataFrame(laporan).T.to_csv(OUT / f'{metode}_classification_report.csv')  # Menyimpan metrik per kelas sebagai CSV.
    display(pd.DataFrame(laporan).T)  # Menampilkan laporan klasifikasi pada notebook.

    selesai = {  # Menyusun hasil akhir metode yang telah diuji.
        'signature': SIGNATURE,  # Menyimpan identitas eksperimen.
        'metrik': hasil,  # Menyimpan seluruh metrik pengujian.
        'epoch_terbaik': int(terbaik['epoch']),  # Menyimpan nomor epoch model terbaik.
    }  # Mengakhiri hasil pengujian.

    result_file.write_text(json.dumps(selesai, indent=2))  # Menyimpan JSON sebagai penanda eksperimen selesai.
    del model  # Menghapus objek model dari memori.
    del opt  # Menghapus objek optimizer dari memori.
    del scaler  # Menghapus objek penskala gradien dari memori.
    gc.collect()  # Meminta Python membebaskan objek yang tidak digunakan.
    torch.cuda.empty_cache()  # Membebaskan cache GPU sebelum pendekatan berikutnya.
    return hasil  # Mengembalikan metrik pengujian akhir.


In [ ]:
hasil_frozen = eksperimen('frozen')  # Melatih frozen backbone dan menyimpan hasilnya agar dapat digunakan kembali.


In [ ]:
riwayat_frozen = pd.read_csv(OUT / 'frozen_loss_dan_metrik_per_epoch.csv')  # Membaca catatan loss frozen dari setiap epoch.
fig, ax = kanvas()  # Membuat kanvas kurva dengan gaya konsisten.
ax.plot(riwayat_frozen['epoch'], riwayat_frozen['train_loss'], marker='o', color=WARNA_GELAP, linewidth=2, label='Loss pelatihan')  # Menggambar training loss.
ax.plot(riwayat_frozen['epoch'], riwayat_frozen['val_loss'], marker='s', color=WARNA_MUDA, linewidth=2, label='Loss validasi')  # Menggambar validation loss.
ax.set_xticks(riwayat_frozen['epoch'])  # Menetapkan nomor epoch pada sumbu horizontal.
ax.set_xlabel('Epoch')  # Menamai sumbu jumlah putaran pelatihan.
ax.set_ylabel('Nilai loss')  # Menamai sumbu nilai kesalahan model.
ax.legend(frameon=False)  # Menampilkan legenda dua kurva.
simpan_gambar(fig, '08_loss_frozen')  # Menampilkan grafik tepat setelah pelatihan frozen.


In [ ]:
fig, ax = kanvas()  # Membuat bidang grafik metrik validasi frozen.
ax.plot(riwayat_frozen['epoch'], riwayat_frozen['val_macro_f1'] * 100, marker='o', color=WARNA_SEDANG, linewidth=2)  # Menggambar F1 makro setiap epoch.
ax.set_xticks(riwayat_frozen['epoch'])  # Menampilkan nomor epoch 1 sampai 5.
ax.set_xlabel('Epoch')  # Menamai sumbu putaran pelatihan.
ax.set_ylabel('F1 makro validasi (%)')  # Menetapkan nilai metrik dalam persen.
ax.set_ylim(0, 100)  # Menggunakan rentang persen yang mudah dibandingkan.
simpan_gambar(fig, '09_f1_makro_validasi_frozen')  # Menampilkan perkembangan validasi frozen.


In [ ]:
def gambar_confusion_matrix(prediksi_df, nama_gambar):  # Membuat confusion matrix dengan format klasik yang seragam.
    matriks = confusion_matrix(prediksi_df['aktual'], prediksi_df['prediksi'], labels=[0, 1])  # Menghitung matriks dari prediksi aktual.
    fig, ax = kanvas(kiri=0.25, bawah=0.22, kanan=0.93, atas=0.95)  # Menyiapkan kanvas standar berukuran 8 × 5 inci.
    ax.set_position([0.34, 0.23, 0.40, 0.64])  # Membuat area matriks lebih ringkas dan berada di tengah.
    ax.grid(False)  # Menghilangkan kisi garis luar supaya matriks bersih.
    ax.imshow(matriks, cmap='Blues', vmin=0, vmax=11000, aspect='equal')  # Menggambar kotak dengan skala warna yang sama.
    ax.axhline(0.5, color='white', linewidth=0.9)  # Memberikan garis pemisah tipis horizontal.
    ax.axvline(0.5, color='white', linewidth=0.9)  # Memberikan garis pemisah tipis vertikal.
    ax.set_xticks([0, 1], labels=NAMA_KELAS)  # Mengisi sumbu prediksi dengan nama dua kelas.
    ax.set_yticks([0, 1], labels=NAMA_KELAS)  # Mengisi sumbu aktual dengan nama dua kelas.
    ax.set_xlabel('Kelas Prediksi')  # Memberi label sumbu horizontal dalam bahasa Indonesia.
    ax.set_ylabel('Kelas Aktual')  # Memberi label sumbu vertikal dalam bahasa Indonesia.
    ax.tick_params(axis='both', length=0)  # Menghilangkan tanda garis kecil pada sumbu.
    for baris in range(2):  # Memeriksa kedua baris kelas aktual.
        for kolom in range(2):  # Memeriksa kedua kolom kelas prediksi.
            nilai = int(matriks[baris, kolom])  # Mengambil jumlah responden pada sel matriks.
            warna_teks = 'white' if nilai > 5500 else WARNA_GELAP  # Memilih kontras tulisan terhadap kotak.
            ax.text(kolom, baris, f'{nilai:,}'.replace(',', '.'), ha='center', va='center', fontsize=13, fontweight='semibold', color=warna_teks)  # Menulis jumlah responden.
    simpan_gambar(fig, nama_gambar)  # Menyimpan PNG dan SVG tanpa judul dalam gambar.

prediksi_frozen = pd.read_csv(OUT / 'frozen_prediksi_uji.csv')  # Membaca hasil pengujian standar frozen dari CSV.
gambar_confusion_matrix(prediksi_frozen, '10_confusion_matrix_frozen_standar')  # Menampilkan matriks frozen setelah pelatihan.


In [ ]:
hasil_qlora = eksperimen('qlora')  # Melatih QLoRA menggunakan subset dan pengaturan evaluasi yang sama.


In [ ]:
riwayat_qlora = pd.read_csv(OUT / 'qlora_loss_dan_metrik_per_epoch.csv')  # Membaca loss QLoRA yang dicatat setiap epoch.
fig, ax = kanvas()  # Membuat kanvas identik dengan grafik frozen.
ax.plot(riwayat_qlora['epoch'], riwayat_qlora['train_loss'], marker='o', color=WARNA_GELAP, linewidth=2, label='Loss pelatihan')  # Menggambar training loss QLoRA.
ax.plot(riwayat_qlora['epoch'], riwayat_qlora['val_loss'], marker='s', color=WARNA_MUDA, linewidth=2, label='Loss validasi')  # Menggambar validation loss QLoRA.
ax.set_xticks(riwayat_qlora['epoch'])  # Menunjukkan nomor epoch yang dilatih.
ax.set_xlabel('Epoch')  # Menamai sumbu putaran pelatihan.
ax.set_ylabel('Nilai loss')  # Menamai sumbu nilai kesalahan.
ax.legend(frameon=False)  # Menampilkan label kedua kurva.
simpan_gambar(fig, '11_loss_qlora')  # Menampilkan kurva QLoRA setelah pelatihan selesai.


In [ ]:
fig, ax = kanvas()  # Membuat kanvas metrik validasi QLoRA.
ax.plot(riwayat_qlora['epoch'], riwayat_qlora['val_macro_f1'] * 100, marker='o', color=WARNA_SEDANG, linewidth=2)  # Menggambar F1 makro QLoRA.
ax.set_xticks(riwayat_qlora['epoch'])  # Menetapkan posisi epoch pada sumbu.
ax.set_xlabel('Epoch')  # Menamai sumbu horizontal.
ax.set_ylabel('F1 makro validasi (%)')  # Menamai nilai dalam bentuk persen.
ax.set_ylim(0, 100)  # Menjaga rentang seragam dengan grafik frozen.
simpan_gambar(fig, '12_f1_makro_validasi_qlora')  # Menampilkan grafik validasi QLoRA.


In [ ]:
prediksi_qlora = pd.read_csv(OUT / 'qlora_prediksi_uji.csv')  # Membaca prediksi standar QLoRA dari CSV.
gambar_confusion_matrix(prediksi_qlora, '13_confusion_matrix_qlora_standar')  # Menggambar matriks QLoRA dengan format yang sama.


In [ ]:
hasil = pd.DataFrame([hasil_frozen, hasil_qlora], index=['Frozen backbone', 'QLoRA'])  # Menggabungkan hasil standar kedua model.
hasil.to_csv(OUT / 'perbandingan_metrik_uji.csv')  # Menyimpan tabel metrik pengujian standar.
display(hasil.round(4))  # Menampilkan hasil standar hingga empat desimal.


In [ ]:
kolom_metrik = ['akurasi', 'macro_f1', 'presisi_cvd', 'recall_cvd', 'f1_cvd']  # Menetapkan lima metrik standar.
nama_metrik = ['Akurasi', 'F1 makro', 'Presisi CVD', 'Recall CVD', 'F1 CVD']  # Menetapkan label grafik.
posisi = np.arange(len(kolom_metrik))  # Menyiapkan posisi setiap metrik.
lebar = 0.36  # Menentukan lebar batang untuk dua pendekatan.
fig, ax = kanvas(bawah=0.25)  # Menyediakan ruang bagi nama metrik.
ax.bar(posisi - lebar / 2, hasil.loc['Frozen backbone', kolom_metrik].astype(float).values * 100, width=lebar, color=WARNA_GELAP, label='Frozen backbone')  # Menggambar metrik frozen.
ax.bar(posisi + lebar / 2, hasil.loc['QLoRA', kolom_metrik].astype(float).values * 100, width=lebar, color=WARNA_MUDA, label='QLoRA')  # Menggambar metrik QLoRA.
ax.set_xticks(posisi, labels=nama_metrik, rotation=12, ha='right')  # Menamai lima metrik secara ringkas.
ax.set_ylabel('Nilai metrik (%)')  # Menetapkan satuan nilai dalam persen.
ax.set_ylim(0, 105)  # Menggunakan rentang persentase yang sama.
ax.legend(frameon=False)  # Menampilkan dua pendekatan pembanding.
simpan_gambar(fig, '14_perbandingan_metrik_standar')  # Menyimpan perbandingan hasil utama.


In [ ]:
kolom_metrik = ['akurasi', 'macro_f1', 'presisi_cvd', 'recall_cvd', 'f1_cvd']  # Menetapkan lima metrik standar.
nama_metrik = ['Akurasi', 'F1 makro', 'Presisi CVD', 'Recall CVD', 'F1 CVD']  # Menetapkan label grafik.
posisi = np.arange(len(kolom_metrik))  # Menyiapkan posisi setiap metrik.
lebar = 0.36  # Menentukan lebar batang untuk dua pendekatan.
fig, ax = kanvas(bawah=0.25)  # Menyediakan ruang bagi nama metrik.

# Menggambar metrik frozen dan qlora, lalu simpan objek batangnya ke dalam variabel
bars_frozen = ax.bar(posisi - lebar / 2, hasil.loc['Frozen backbone', kolom_metrik].astype(float).values * 100, width=lebar, color=WARNA_GELAP, label='Frozen backbone')
bars_qlora = ax.bar(posisi + lebar / 2, hasil.loc['QLoRA', kolom_metrik].astype(float).values * 100, width=lebar, color=WARNA_MUDA, label='QLoRA')

# Menambahkan label nilai di atas masing-masing batang (format 2 angka di belakang koma)
ax.bar_label(bars_frozen, fmt='%.1f%%', padding=3, fontsize=8, rotation=0)
ax.bar_label(bars_qlora, fmt='%.1f%%', padding=3, fontsize=8, rotation=0)

ax.set_xticks(posisi, labels=nama_metrik, rotation=12, ha='right')  # Menamai lima metrik secara ringkas.
ax.set_ylabel('Nilai metrik (%)')  # Menetapkan satuan nilai dalam persen.
ax.set_ylim(0, 112)  # Menggunakan rentang persentase yang sama dan sedikit diperluas ke atas agar label tertinggi tidak terpotong.
ax.legend(frameon=False)  # Menampilkan dua pendekatan pembanding.
simpan_gambar(fig, '14_perbandingan_metrik_standar')  # Menyimpan perbandingan hasil utama.

In [ ]:
total_frozen = riwayat_frozen['durasi_total_menit'].sum()  # Menjumlahkan waktu frozen dari seluruh epoch.
total_qlora = riwayat_qlora['durasi_total_menit'].sum()  # Menjumlahkan waktu QLoRA dari seluruh epoch.
tabel_durasi = pd.DataFrame({'Metode': ['Frozen backbone', 'QLoRA'], 'Durasi (menit)': [total_frozen, total_qlora]})  # Membuat tabel biaya komputasi.
tabel_durasi.to_csv(OUT / 'perbandingan_durasi.csv', index=False)  # Menyimpan durasi eksperimen.
display(tabel_durasi.round(2))  # Menampilkan tabel durasi dalam menit.
fig, ax = kanvas(bawah=0.18)  # Membuat kanvas durasi dengan ukuran seragam.
ax.bar(tabel_durasi['Metode'], tabel_durasi['Durasi (menit)'], color=[WARNA_GELAP, WARNA_MUDA], width=0.60)  # Menampilkan waktu tiap pendekatan.
ax.set_ylabel('Durasi pelatihan dan validasi (menit)')  # Menjelaskan arti angka durasi.

simpan_gambar(fig, '15_perbandingan_durasi')  # Menampilkan grafik durasi pelatihan.

In [ ]:
total_frozen = riwayat_frozen['durasi_total_menit'].sum()  # Menjumlahkan waktu frozen dari seluruh epoch.
total_qlora = riwayat_qlora['durasi_total_menit'].sum()  # Menjumlahkan waktu QLoRA dari seluruh epoch.
tabel_durasi = pd.DataFrame({'Metode': ['Frozen backbone', 'QLoRA'], 'Durasi (menit)': [total_frozen, total_qlora]})  # Membuat tabel biaya komputasi.
tabel_durasi.to_csv(OUT / 'perbandingan_durasi.csv', index=False)  # Menyimpan durasi eksperimen.
display(tabel_durasi.round(2))  # Menampilkan tabel durasi dalam menit.

fig, ax = kanvas(bawah=0.18)  # Membuat kanvas durasi dengan ukuran seragam.
bars = ax.bar(tabel_durasi['Metode'], tabel_durasi['Durasi (menit)'], color=[WARNA_GELAP, WARNA_MUDA], width=0.60)  # Menampilkan waktu tiap pendekatan.

# Mengonversi nilai desimal menit ke format "xx menit xx detik" untuk setiap batang
labels_durasi = []
for m in tabel_durasi['Durasi (menit)']:
    menit = int(m)
    detik = int(round((m - menit) * 60))
    if detik == 60:
        menit += 1
        detik = 0
    labels_durasi.append(f"{menit} menit {detik} detik")

# Menampilkan teks "xx menit xx detik" di atas masing-masing batang
ax.bar_label(bars, labels=labels_durasi, padding=4, fontsize=9, fontweight='bold')

ax.set_ylabel('Durasi pelatihan dan validasi (menit)')  # Menjelaskan arti angka durasi.

# Menyesuaikan batas sumbu Y secara otomatis agar ada ruang untuk teks di atas batang tertinggi
max_durasi = tabel_durasi['Durasi (menit)'].max()
ax.set_ylim(0, max_durasi * 1.25)

simpan_gambar(fig, '15_perbandingan_durasi')  # Menampilkan grafik durasi pelatihan.

In [ ]:
def tuning_threshold(metode):  # Mengoptimalkan ambang keputusan berdasarkan validasi tanpa melatih ulang model.
    lokasi_bobot = OUT / f'{metode}_bobot_terbaik.pt'  # Menentukan checkpoint terbaik untuk satu pendekatan.
    assert lokasi_bobot.exists(), f'Checkpoint tidak ditemukan: {lokasi_bobot}'  # Memastikan bobot tersedia sebelum inferensi.
    checkpoint = torch.load(lokasi_bobot, map_location='cpu', weights_only=True)  # Membaca bobot yang dipilih berdasarkan validasi.
    assert checkpoint['signature'] == SIGNATURE, 'Checkpoint tidak cocok dengan penelitian saat ini.'  # Memeriksa identitas eksperimen.

    model = buat_model(metode)  # Membangun ulang arsitektur model sesuai pendekatan.
    pulihkan_bobot(model, checkpoint['bobot'])  # Mengembalikan parameter trainable terbaik.
    model.eval()  # Menonaktifkan dropout pada waktu inferensi.

    def inferensi_probabilitas(loader):  # Menghasilkan probabilitas dan prediksi argmax secara bersamaan.
        aktual = []  # Menyediakan penampung label sebenarnya.
        probabilitas = []  # Menyediakan penampung probabilitas kelas CVD.
        prediksi_argmax = []  # Menyediakan penampung prediksi standar model.
        with torch.inference_mode():  # Menonaktifkan pembentukan gradien pada GPU.
            for x, mask, y in tqdm(loader, desc=f'Inferensi {metode}', unit='batch'):  # Menampilkan progres tiap batch inferensi.
                x = x.cuda(non_blocking=True)  # Memindahkan token ke GPU.
                mask = mask.cuda(non_blocking=True)  # Memindahkan attention mask ke GPU.
                with torch.autocast('cuda', dtype=torch.float16):  # Menjalankan inferensi pada FP16.
                    keluaran = model(input_ids=x, attention_mask=mask)  # Mengambil dua nilai logit kelas.
                peluang = torch.softmax(keluaran.logits.float(), dim=-1)[:, 1]  # Menghitung probabilitas kelas CVD.
                label_prediksi = keluaran.logits.argmax(dim=-1)  # Menghitung keputusan standar dengan argmax.
                aktual.extend(y.numpy().tolist())  # Menyimpan label sebenarnya dari CPU.
                probabilitas.extend(peluang.cpu().numpy().tolist())  # Memindahkan probabilitas ke CPU.
                prediksi_argmax.extend(label_prediksi.cpu().numpy().tolist())  # Memindahkan prediksi standar ke CPU.
        return np.array(aktual), np.array(probabilitas), np.array(prediksi_argmax)  # Mengembalikan tiga array inferensi.

    y_val, p_val, _ = inferensi_probabilitas(loader_val)  # Memperoleh keluaran data validasi.
    skor_kandidat = []  # Menampung hasil pencarian tiap batas keputusan.
    for nomor in range(5, 96):  # Menguji nilai 0,05 sampai 0,95 dengan langkah 0,01.
        ambang = nomor / 100  # Mengubah bilangan bulat menjadi ambang probabilitas.
        pred_val = (p_val > ambang).astype(int)  # Menggunakan operator lebih besar agar seri 0,50 memilih kelas 0.
        skor = f1_score(y_val, pred_val, average='macro', zero_division=0)  # Menghitung F1 makro validasi.
        presisi_val, recall_val, f1_val, _ = precision_recall_fscore_support(y_val, pred_val, labels=[1], zero_division=0)  # Menghitung metrik kelas CVD pada validasi.
        skor_kandidat.append({  # Mencatat hasil tiap batas keputusan untuk analisis presisi dan recall.
            'threshold': ambang,  # Menyimpan kandidat ambang.
            'macro_f1_validasi': skor,  # Menyimpan F1 makro pada data validasi.
            'presisi_cvd_validasi': float(presisi_val[0]),  # Mencatat presisi kelas CVD.
            'recall_cvd_validasi': float(recall_val[0]),  # Mencatat recall kelas CVD.
            'f1_cvd_validasi': float(f1_val[0]),  # Mencatat F1 kelas CVD.
        })  # Mengakhiri catatan satu kandidat threshold.

    tabel = pd.DataFrame(skor_kandidat)  # Menyusun seluruh kurva hasil validasi.
    tabel['jarak_050'] = (tabel['threshold'] - 0.50).abs()  # Mengatur kriteria seri terdekat ke 0,50.
    tabel_urut = tabel.sort_values(['macro_f1_validasi', 'jarak_050'], ascending=[False, True])  # Mengurutkan kandidat berdasarkan performa validasi.
    ambang_terbaik = float(tabel_urut.iloc[0]['threshold'])  # Menetapkan satu ambang pilihan untuk metode ini.
    tabel = tabel.drop(columns='jarak_050')  # Menghapus kolom bantu yang tidak diperlukan dalam grafik.
    tabel.to_csv(OUT / f'{metode}_threshold_validasi.csv', index=False)  # Menyimpan nilai kurva untuk Bab IV.

    y_uji, p_uji, pred_standar = inferensi_probabilitas(loader_uji)  # Menghasilkan probabilitas uji setelah ambang dipilih.
    pred_tuning = (p_uji > ambang_terbaik).astype(int)  # Menerapkan threshold pilihan validasi pada data uji.
    pred_050 = (p_uji > 0.50).astype(int)  # Menghasilkan threshold 0,50 dengan aturan seri yang konsisten.
    ketidakcocokan = int(np.sum(pred_050 != pred_standar))  # Menghitung kemungkinan selisih karena pembulatan softmax.
    print('Perbedaan aturan 0,50 dan argmax:', ketidakcocokan)  # Melaporkan hasil audit agar tidak tersembunyi.

    sumber_standar = pd.read_csv(OUT / f'{metode}_prediksi_uji.csv')  # Membaca hasil evaluasi standar dari tahap pelatihan.
    assert np.array_equal(y_uji, sumber_standar['aktual'].to_numpy()), 'Urutan data uji berbeda.'  # Memastikan responden uji identik.
    assert np.array_equal(pred_standar, sumber_standar['prediksi'].to_numpy()), 'Prediksi argmax berbeda dari evaluasi standar.'  # Mencegah pembandingan yang tidak konsisten.

    ringkasan = pd.DataFrame([metrik(y_uji, pred_standar), metrik(y_uji, pred_tuning)], index=['Standar argmax', f'Tuning {ambang_terbaik:.2f}'])  # Membandingkan dua aturan keputusan.
    ringkasan.to_csv(OUT / f'{metode}_threshold_ringkasan_uji.csv')  # Menyimpan metrik standar dan penyesuaian ambang.
    prediksi = pd.DataFrame({'aktual': y_uji, 'prediksi': pred_tuning, 'prediksi_standar': pred_standar, 'probabilitas_cvd': p_uji})  # Mencatat semua prediksi uji.
    prediksi.to_csv(OUT / f'{metode}_threshold_prediksi_uji.csv', index=False)  # Menyimpan hasil responden pada CSV.
    print('Threshold pilihan validasi:', round(ambang_terbaik, 2))  # Menampilkan ambang terbaik.
    display(ringkasan.round(4))  # Menampilkan perbandingan metrik.

    del model  # Menghapus model setelah inferensi selesai.
    gc.collect()  # Membersihkan objek Python yang tidak lagi diperlukan.
    torch.cuda.empty_cache()  # Mengurangi cache GPU yang tidak terpakai.
    return {'threshold': ambang_terbaik, 'metrik': ringkasan.loc[f'Tuning {ambang_terbaik:.2f}'].to_dict()}  # Mengembalikan ambang dan metrik.


In [ ]:
hasil_threshold_frozen = tuning_threshold('frozen')  # Menentukan ambang frozen dari validasi dan menguji pada data uji.


In [ ]:
kurva_frozen = pd.read_csv(OUT / 'frozen_threshold_validasi.csv')  # Membaca nilai F1 makro seluruh kandidat threshold frozen.
fig, ax = kanvas()  # Membuat grafik kurva dengan format seragam.

# Menggambar hasil validasi per ambang (dikalikan 100 untuk persen)
ax.plot(kurva_frozen['threshold'], kurva_frozen['macro_f1_validasi'] * 100, color=WARNA_GELAP, linewidth=2, label='F1 Makro Validasi')

# Menandai batas keputusan terpilih.
ax.axvline(hasil_threshold_frozen['threshold'], color=WARNA_SEDANG, linestyle='--', label=f"Ambang terpilih: {hasil_threshold_frozen['threshold']:.2f}")

# Mengambil nilai F1 makro pada threshold terpilih
th_val = hasil_threshold_frozen['threshold']
row_terpilih = kurva_frozen.loc[kurva_frozen['threshold'] == th_val].iloc[0]
f1_val = row_terpilih['macro_f1_validasi'] * 100

# Menandai titik pada kurva
ax.scatter([th_val], [f1_val], color=WARNA_GELAP, s=50, zorder=5)

# Style kotak latar belakang putih agar teks jelas terbaca
bbox_style = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9)

# Menempatkan teks nilai di sebelah kiri titik dengan sedikit geseran ke atas (+ 4)
ax.text(th_val - 0.03, f1_val + 4, f'{f1_val:.2f}%', color=WARNA_GELAP, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)

ax.set_xlabel('Ambang keputusan kelas CVD History')  # Menamai sumbu horizontal dalam bahasa Indonesia.
ax.set_ylabel('F1 makro validasi (%)')  # Menjelaskan besaran sumbu vertikal.
ax.set_ylim(0, 105)  # Menentukan skala persentase dengan ruang di atas.
ax.legend(frameon=False)  # Menampilkan batas keputusan terpilih.
simpan_gambar(fig, '16_threshold_validasi_frozen')  # Menampilkan kurva sesudah tuning frozen.

In [ ]:
kurva_pr_frozen = pd.read_csv(OUT / 'frozen_threshold_validasi.csv')  # Membaca hasil validasi dari seluruh kandidat ambang.
fig, ax = kanvas()  # Membuat grafik presisi dan recall dengan ukuran standar.

# Menampilkan perubahan presisi dan recall saat ambang berubah (dikalikan 100 untuk persen)
ax.plot(kurva_pr_frozen['threshold'], kurva_pr_frozen['presisi_cvd_validasi'] * 100, color=WARNA_GELAP, linewidth=2, label='Presisi CVD')
ax.plot(kurva_pr_frozen['threshold'], kurva_pr_frozen['recall_cvd_validasi'] * 100, color=WARNA_MUDA, linewidth=2, label='Recall CVD')

# Menandai garis vertikal ambang berdasarkan F1 makro validasi.
ax.axvline(hasil_threshold_frozen['threshold'], color=WARNA_SEDANG, linestyle='--', linewidth=1, label='Ambang terpilih')

# Mengambil nilai presisi dan recall pada threshold terpilih
th_val = hasil_threshold_frozen['threshold']
row_terpilih = kurva_pr_frozen.loc[kurva_pr_frozen['threshold'] == th_val].iloc[0]
p_val = row_terpilih['presisi_cvd_validasi'] * 100
r_val = row_terpilih['recall_cvd_validasi'] * 100

# Menandai titik pada kurva
ax.scatter([th_val, th_val], [p_val, r_val], color=[WARNA_GELAP, WARNA_MUDA], s=50, zorder=5)

# Style kotak latar belakang putih agar teks jelas terbaca tanpa panah
bbox_style = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9)

# Menempatkan teks dengan jarak vertikal yang lebih lega agar tidak bertumpuk
ax.text(th_val - 0.03, p_val - 13, f'{p_val:.2f}%', color=WARNA_GELAP, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)
ax.text(th_val - 0.03, r_val + 20, f'{r_val:.2f}%', color=WARNA_MUDA, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)

ax.set_xlabel('Ambang keputusan kelas CVD History')  # Menamai sumbu horizontal.
ax.set_ylabel('Nilai metrik validasi (%)')  # Menamai sumbu presisi dan recall dalam persen.
ax.set_ylim(0, 105)  # Menentukan skala persentase (diberi ruang di atas).
ax.legend(frameon=False)  # Menampilkan legenda dua metrik dan ambang.
simpan_gambar(fig, '17_kurva_presisi_recall_frozen')  # Menyimpan dan menampilkan grafik segera setelah threshold tuning.

In [ ]:
pred_tuning_frozen = pd.read_csv(OUT / 'frozen_threshold_prediksi_uji.csv')  # Membaca prediksi setelah tuning frozen.
gambar_confusion_matrix(pred_tuning_frozen, '18_confusion_matrix_frozen_tuning')  # Menampilkan matriks keputusan yang sudah disesuaikan.


In [ ]:
hasil_threshold_qlora = tuning_threshold('qlora')  # Menentukan ambang QLoRA dari validasi dan menguji pada data uji.


In [ ]:
kurva_qlora = pd.read_csv(OUT / 'qlora_threshold_validasi.csv')  # Membaca nilai F1 makro seluruh kandidat threshold QLoRA.
fig, ax = kanvas()  # Membuat bidang kurva sama seperti model frozen.

# Menggambar hubungan ambang dengan F1 makro (dikalikan 100 untuk persen)
ax.plot(kurva_qlora['threshold'], kurva_qlora['macro_f1_validasi'] * 100, color=WARNA_GELAP, linewidth=2, label='F1 Makro Validasi')

# Menandai batas terpilih QLoRA.
ax.axvline(hasil_threshold_qlora['threshold'], color=WARNA_SEDANG, linestyle='--', label=f"Ambang terpilih: {hasil_threshold_qlora['threshold']:.2f}")

# Mengambil nilai F1 makro pada threshold terpilih
th_val = hasil_threshold_qlora['threshold']
row_terpilih = kurva_qlora.loc[kurva_qlora['threshold'] == th_val].iloc[0]
f1_val = row_terpilih['macro_f1_validasi'] * 100

# Menandai titik pada kurva
ax.scatter([th_val], [f1_val], color=WARNA_GELAP, s=50, zorder=5)

# Style kotak latar belakang putih agar teks jelas terbaca
bbox_style = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9)

# Menggeser posisi teks sedikit ke atas (+ 4 pada sumbu Y) dan ke kiri (- 0.03 pada sumbu X)
ax.text(th_val - 0.03, f1_val + 4, f'{f1_val:.2f}%', color=WARNA_GELAP, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)

ax.set_xlabel('Ambang keputusan kelas CVD History')  # Menetapkan nama sumbu probabilitas.
ax.set_ylabel('F1 makro validasi (%)')  # Menetapkan nama sumbu skor validasi.
ax.set_ylim(0, 105)  # Menentukan skala persentase.
ax.legend(frameon=False)  # Memberi keterangan untuk garis ambang terpilih.
simpan_gambar(fig, '19_threshold_validasi_qlora')  # Menampilkan kurva sesudah tuning QLoRA.

In [ ]:
kurva_pr_qlora = pd.read_csv(OUT / 'qlora_threshold_validasi.csv')  # Membaca hasil validasi dari seluruh kandidat ambang.
fig, ax = kanvas()  # Membuat grafik presisi dan recall dengan ukuran standar.

# Menampilkan perubahan presisi dan recall saat ambang berubah (dikalikan 100 untuk persen)
ax.plot(kurva_pr_qlora['threshold'], kurva_pr_qlora['presisi_cvd_validasi'] * 100, color=WARNA_GELAP, linewidth=2, label='Presisi CVD')
ax.plot(kurva_pr_qlora['threshold'], kurva_pr_qlora['recall_cvd_validasi'] * 100, color=WARNA_MUDA, linewidth=2, label='Recall CVD')

# Menandai garis vertikal ambang berdasarkan F1 makro validasi.
ax.axvline(hasil_threshold_qlora['threshold'], color=WARNA_SEDANG, linestyle='--', linewidth=1, label='Ambang terpilih')

# Mengambil nilai presisi dan recall pada threshold terpilih
th_val = hasil_threshold_qlora['threshold']
row_terpilih = kurva_pr_qlora.loc[kurva_pr_qlora['threshold'] == th_val].iloc[0]
p_val = row_terpilih['presisi_cvd_validasi'] * 100
r_val = row_terpilih['recall_cvd_validasi'] * 100

# Menandai titik dengan scatter plot
ax.scatter([th_val, th_val], [p_val, r_val], color=[WARNA_GELAP, WARNA_MUDA], s=50, zorder=5)

# Style kotak latar belakang putih agar teks tetap jelas terbaca tanpa garis kurva di belakangnya
bbox_style = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9)

# Menempatkan teks langsung di sebelah kiri titik tanpa panah
ax.text(th_val - 0.03, p_val, f'{p_val:.2f}%', color=WARNA_GELAP, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)
ax.text(th_val - 0.03, r_val, f'{r_val:.2f}%', color=WARNA_MUDA, fontsize=9, fontweight='bold', va='center', ha='right', bbox=bbox_style)

ax.set_xlabel('Ambang keputusan kelas CVD History')  # Menamai sumbu horizontal.
ax.set_ylabel('Nilai metrik validasi (%)')  # Menamai sumbu presisi dan recall dalam persen.
ax.set_ylim(0, 105)  # Menentukan skala persentase.
ax.legend(frameon=False)  # Menampilkan legenda dua metrik dan ambang.
simpan_gambar(fig, '20_kurva_presisi_recall_qlora')  # Menyimpan dan menampilkan grafik segera setelah threshold tuning.

In [ ]:
pred_tuning_qlora = pd.read_csv(OUT / 'qlora_threshold_prediksi_uji.csv')  # Membaca prediksi setelah tuning QLoRA.
gambar_confusion_matrix(pred_tuning_qlora, '21_confusion_matrix_qlora_tuning')  # Menampilkan confusion matrix tuning QLoRA.


In [ ]:
ringkasan_tuning_frozen = pd.read_csv(OUT / 'frozen_threshold_ringkasan_uji.csv', index_col=0)  # Membaca hasil threshold frozen.
ringkasan_tuning_qlora = pd.read_csv(OUT / 'qlora_threshold_ringkasan_uji.csv', index_col=0)  # Membaca hasil threshold QLoRA.
tabel_tuning = pd.DataFrame([ringkasan_tuning_frozen.iloc[-1], ringkasan_tuning_qlora.iloc[-1]], index=['Frozen backbone', 'QLoRA'])  # Menyatukan hasil tuning kedua model.
tabel_tuning.to_csv(OUT / 'perbandingan_metrik_setelah_tuning.csv')  # Menyimpan ringkasan evaluasi tuning.
display(tabel_tuning.round(4))  # Menampilkan metrik sesudah penyesuaian ambang.


In [ ]:
nama_batang = ['Frozen standar', 'Frozen tuning', 'QLoRA standar', 'QLoRA tuning']  # Menentukan empat kondisi evaluasi.
nilai_batang = [hasil.loc['Frozen backbone', 'macro_f1'], tabel_tuning.loc['Frozen backbone', 'macro_f1'], hasil.loc['QLoRA', 'macro_f1'], tabel_tuning.loc['QLoRA', 'macro_f1']]  # Mengambil F1 makro setiap kondisi.

fig, ax = kanvas(bawah=0.20)  # Membuat bidang grafik perbandingan akhir.
bars = ax.bar(nama_batang, np.asarray(nilai_batang) * 100, color=[WARNA_GELAP, WARNA_MUDA, WARNA_SEDANG, WARNA_PUCAT], width=0.65)  # Menampilkan seluruh nilai F1 makro.

# Menambahkan label nilai di atas batang diagram dengan format 2 angka di belakang koma
ax.bar_label(bars, fmt='%.2f%%', padding=3)

ax.set_ylabel('F1 makro pengujian (%)')  # Menjelaskan satuan nilai pada sumbu vertikal.
ax.set_ylim(0, 105)  # Menjaga rentang metrik dan memberi ruang untuk label di atas.
simpan_gambar(fig, '22_perbandingan_f1_makro_threshold')  # Menampilkan grafik perbandingan threshold.

In [ ]:
metadata = {  # Menyusun dokumentasi konfigurasi dan ukuran data aktual.
    'signature': SIGNATURE,  # Mencatat identitas eksperimen agar hasil tidak tertukar.
    'data_file': str(FILE),  # Mencatat lokasi dataset sumber.
    'model': MODEL,  # Mencatat checkpoint dasar TinyLlama.
    'gpu': torch.cuda.get_device_name(0),  # Mencatat perangkat komputasi.
    'data_awal': int(jumlah_awal),  # Mencatat jumlah responden dalam XPT.
    'kolom_awal': int(jumlah_kolom),  # Mencatat jumlah variabel awal.
    'data_bersih': len(bersih),  # Mencatat responden yang lolos preprocessing.
    'sampel_eksperimen': len(sampel),  # Mencatat jumlah sampel terpilih.
    'metode_sampling': 'Stratified random sampling',  # Mencatat teknik pemilihan sampel.
    'profil_unik_sampel': int(jumlah_profil),  # Mencatat kombinasi fitur berbeda dalam sampel.
    'profil_berulang_data_bersih': int(profil_berulang_penuh),  # Mencatat pengulangan profil sebelum sampling.
    'profil_berulang_sampel': int(ulang_fitur),  # Mencatat pengulangan profil sesudah sampling.
    'profil_dengan_label_berbeda': int(konflik_label),  # Mencatat profil yang memiliki dua label.
    'n_latih_sebelum_rus': len(latih),  # Mencatat ukuran data latih sebelum diseimbangkan.
    'n_latih_setelah_rus': len(latih_rus),  # Mencatat ukuran data latih setelah RUS.
    'n_validasi': len(validasi),  # Mencatat ukuran data validasi.
    'n_uji': len(uji),  # Mencatat ukuran data pengujian.
    'split': 'Pemilihan kandidat GroupShuffleSplit berdasarkan 11 fitur',  # Mencatat aturan pemisahan kelompok.
    'jumlah_kandidat_split': JUMLAH_KANDIDAT,  # Menjelaskan banyak pembagian acak yang dievaluasi.
    'prevalensi_sampel': float(prevalensi),  # Mencatat persentase positif yang menjadi acuan pembagian.
    'identitas_split': kunci_split,  # Mencatat penanda yang memastikan pembagian konsisten.
    'rasio_aktual': ringkasan_split.to_dict(orient='records'),  # Mencatat ukuran serta distribusi setiap subset.
    'rus': '1:1 hanya pada data latih',  # Mencatat penanganan ketidakseimbangan kelas.
    'fitur': FITUR,  # Mencatat seluruh fitur yang menjadi masukan model.
    'target': '_MICHD',  # Mencatat target status riwayat CHD/MI.
    'label_positif': 'CVD History',  # Mencatat interpretasi label 1.
    'seed': SEED,  # Mencatat seed yang digunakan.
    'epoch': EPOCHS,  # Mencatat maksimum epoch pelatihan.
    'batch_latih': BATCH,  # Mencatat batch size pelatihan.
    'batch_evaluasi': BATCH_EVAL,  # Mencatat batch size evaluasi.
    'max_token': MAX_TOKEN,  # Mencatat batas token yang direncanakan.
    'panjang_token_teramati': panjang_max,  # Mencatat panjang token terbesar yang diperiksa.
    'qlora': {  # Mencatat pengaturan khusus adapter QLoRA.
        'r': 64,  # Mencatat nilai rank LoRA.
        'alpha': 32,  # Mencatat nilai alpha LoRA.
        'dropout': 0.05,  # Mencatat dropout adapter.
        'modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # Mencatat modul attention yang diadaptasi.
        'quantization': 'NF4 4-bit',  # Mencatat skema kuantisasi model dasar.
    },  # Mengakhiri konfigurasi QLoRA.
    'metrik': hasil.to_dict(),  # Mencatat hasil pengujian kedua pendekatan.
    'threshold_terpilih': {'frozen': hasil_threshold_frozen['threshold'], 'qlora': hasil_threshold_qlora['threshold']},  # Mencatat ambang pilihan validasi.
    'metrik_setelah_tuning': tabel_tuning.to_dict(),  # Mencatat hasil pengujian tambahan setelah tuning.
}  # Mengakhiri metadata penelitian.

file_metadata = OUT / 'metadata_eksperimen.json'  # Menentukan file penyimpanan metadata.
file_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))  # Menyimpan seluruh metadata dalam JSON.
print('Lokasi hasil penelitian:', OUT)  # Menampilkan folder keluaran yang berisi CSV dan PNG.


In [ ]:
ringkasan_berkas = sorted([p.name for p in FOLDER_GAMBAR.glob('*.png')])  # Mengumpulkan nama semua gambar Bab IV.
print('Total gambar PNG:', len(ringkasan_berkas))  # Menampilkan banyaknya grafik hasil penelitian.
print('Folder gambar:', FOLDER_GAMBAR)  # Menampilkan lokasi gambar untuk ditempel pada tesis.
print('Folder metadata dan CSV:', OUT)  # Menampilkan lokasi data pendukung yang tersimpan di Drive.
